# iSign English → Sign Pose — pre-training readiness (controlled experiment)

Gets the iSign dataset and the Progressive Transformers SLP model to the point where a **controlled training experiment** can start. It does **not** start full training. Every check records PASS / FAIL / UNVERIFIED, and the last section prints **READY FOR CONTROLLED TRAINING** or **NOT READY** with the exact blockers.

**Changes relative to the earlier notebook** (details in `CHANGELOG.md`):

1. **Landmark-level validity** — confidence is kept; missing / zero-filled / low-confidence / non-finite landmarks are flagged separately and excluded from the loss together with padded frames.
2. **Real splits** — train / val / test split **by video ID** (80/10/10, fixed seed); the 5-example pilot is isolated.
3. **Coordinate normalisation is investigated, not assumed** — distributions, out-of-bounds rates, hand/body coordinate compatibility, z scale, discontinuities; parameters are saved and hashed.
4. **Explicit device handling** — everything is placed on one device and verified; the CUDA step is only marked PASS on a real GPU.
5. **Overfit experiment** on a small fixed training subset.
6. **Training / evaluation foundation** — training loop, validation, best checkpoint, early stopping, resume, guarded test evaluation.

**Frame convention (unchanged):** 75 landmarks = 33 body + 21 left hand + 21 right hand (pose-format indices `0:33` and `501:543`); 225 coordinates `[x0,y0,z0,x1,…]` + 1 frame counter = **226 values per frame**; counter = `frame_index / n_frames`.

**Run order:** top to bottom on a **T4 GPU runtime** (Runtime → Change runtime type). On a CPU runtime everything except the CUDA checks still runs, and those are reported as UNVERIFIED. The large dataset stays on Drive and is read through the virtual concatenation of the archive parts; nothing is combined or extracted.

**Switches to know about** (cell 1): `FULL_CONVERT` (convert every shard — hours), `RUN_FULL_TRAINING` and `RUN_TEST_EVAL` (both `False`; they are only honoured if the decision is READY).

## 1. Setup and settings

In [ ]:
# CELL 1 — Mount Drive, define folders and every setting in one place.
import copy
import gc
import importlib
import itertools
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

# ---------- Dataset on Drive: read in place, never copied or combined ----------
DATA_DIR = Path("/content/drive/MyDrive/ISL_Project/iSign")
CSV_PATH = DATA_DIR / "iSign_v1.1.csv"
POSE_DIR = DATA_DIR / "pose_parts"
POSE_PART_NAMES = [f"iSign-poses_v1.1_part_{s}" for s in ("aa", "ab", "ac", "ad")]

# ---------- Local scratch: code and small artefacts only ----------
CODE_DIR = Path("/content/isl_code")
REPO_URL = "https://github.com/BenSaunders27/ProgressiveTransformersSLP.git"
REPO_DIR = Path("/content/ProgressiveTransformersSLP")
WORK_DIR = Path("/content/isl_work")

# ---------- Derived data and reports that must survive a Colab reset (your private Drive) ----------
# Converted shards are derived from iSign. Keep them private: the dataset licence forbids redistribution.
DRIVE_OUT = Path("/content/drive/MyDrive/ISL_Project/pre_training_ready")
PROC_PARENT = DRIVE_OUT / "processed"

for _d in (CODE_DIR, WORK_DIR, DRIVE_OUT, PROC_PARENT):
    _d.mkdir(parents=True, exist_ok=True)

# ---------- Reproducibility and splitting ----------
SEED = 27
SPLITS = ("train", "val", "test")
SPLIT_FRACTIONS = (0.8, 0.1, 0.1)      # by VIDEO, so sequence shares are approximate
MIN_MATCH_RATE = 0.9                   # share of CSV rows that must have a pose file

# ---------- Conversion ----------
SHARD_SIZE = 256
CONVERT_MAX_SHARDS = {"train": 4, "val": 1, "test": 1}   # review-scale conversion (≈1 000 / 250 / 250 sequences)
FULL_CONVERT = False                   # True converts EVERY shard (reads ~all of the archive; takes hours; resumable)
STORAGE_DTYPE = "float16"              # on-disk precision of the coordinates (error is checked)
QUALITY = {"min_frames": 2, "max_frames": 600, "min_valid_fraction": 0.15,
           "max_empty_frame_fraction": 0.5, "require_hands": True}
PE_MAX_LEN = 5000                      # positional-encoding length; must exceed QUALITY["max_frames"]

# ---------- Normalisation / validity rules (see the diagnostics section) ----------
CONF_THRESHOLD = 0.5                   # landmarks below this confidence are invalid
ZERO_FILL_IS_MISSING = True            # an exact raw (0, 0, 0) landmark is the file format's fill value -> invalid
Z_MODE = "keep"                        # "keep" = leave z in source units; "scale" = divide by per-component train-sample scales
DIAG_N_SEQUENCES = 300                 # training-split sample used for the coordinate diagnostics
PILOT_N = 5

# ---------- Model / training ----------
BATCH_SIZE = 8                         # conservative for a 16 GB T4; raise once memory use is known
PILOT_BATCH_SIZE = 2
NUM_WORKERS = 2
MIN_FREE_GB = 8                        # the conversion stops cleanly (and resumes later) if the local disk has less free space
LEARNING_RATE = 1e-3
MAX_EPOCHS = 100
EARLY_STOP_PATIENCE = 10               # validations without improvement
REQUIRE_GPU_FOR_TRAINING = True

# ---------- Overfit experiment ----------
OVERFIT_N = 32
OVERFIT_STEPS = 300
OVERFIT_MAX_FRAMES = 250
OVERFIT_BATCH = 8

# ---------- Safety switches (both stay False until you have reviewed the report) ----------
RUN_FULL_TRAINING = False
RUN_TEST_EVAL = False

assert QUALITY["max_frames"] < PE_MAX_LEN, "PE_MAX_LEN must exceed the longest allowed sequence"
assert Z_MODE in ("keep", "scale")
print("Settings loaded. Dataset stays on Drive; outputs go to", DRIVE_OUT)

## 2. File transfer to Drive — already done, commented out
Original Hugging Face → Drive transfer, kept verbatim but commented out. Select the lines and press `Ctrl+/` to re-enable.

In [ ]:
# CELLS 3-6 (original): Hugging Face login and metadata download
# ALREADY DONE: the dataset is on Drive. Everything below is commented out on purpose;
# select all lines and press Ctrl+/ to re-enable. (Original code, unchanged.)

# # CELL 3 — Install/upgrade the Hugging Face Hub client.
# !pip install -U huggingface_hub
#

# # CELL 4 — Log in to Hugging Face so gated iSign files can be accessed.
# from huggingface_hub import notebook_login
# notebook_login()
#

# # CELL 5 — List files published in the iSign dataset repository.
#
# from huggingface_hub import HfApi
#
# api = HfApi()
# repo_id = "Exploration-Lab/iSign"
#
# info = api.dataset_info(
#     repo_id,
#     files_metadata=True
# )
#
# for f in info.siblings:
#     size = getattr(f, "size", None)
#     size_gb = f"{size / 1e9:.2f} GB" if size is not None else "size unknown"
#     print(f"{size_gb:>12} | {f.rfilename}")
#

# # CELL 6 — Download the iSign CSV metadata files to Google Drive.
# from huggingface_hub import hf_hub_download
# from pathlib import Path
# import shutil
#
# DRIVE_DIR = Path("/content/drive/MyDrive/ISL_Project/iSign")
# DRIVE_DIR.mkdir(parents=True, exist_ok=True)
#
# repo_id = "Exploration-Lab/iSign"
#
# files = [
#     "iSign_v1.1.csv",
#     "word-description-dataset_v1.1.csv",
#     "word-presence-dataset_v1.1.csv",
# ]
#
# for filename in files:
#     downloaded = hf_hub_download(
#         repo_id=repo_id,
#         filename=filename,
#         repo_type="dataset"
#     )
#
#     destination = DRIVE_DIR / filename
#     shutil.copy2(downloaded, destination)
#     print(f"Saved: {destination}")
#

In [ ]:
# CELL 10 (original): transfer the large pose/video parts from Hugging Face to Drive
# ALREADY DONE: the dataset is on Drive. Everything below is commented out on purpose;
# select all lines and press Ctrl+/ to re-enable. (Original code, unchanged.)

# # CELL 10 — Transfer the large gated iSign pose/video parts from Hugging Face to Drive.
# # One-cell pipeline: download one file -> upload to Drive -> verify -> clear local cache -> next file
#
# from pathlib import Path
# import shutil
# import time
# from huggingface_hub import HfApi, hf_hub_download
# from google.colab import auth
# from googleapiclient.discovery import build
# from googleapiclient.http import MediaFileUpload
#
# REPO_ID = "Exploration-Lab/iSign"
# REPO_TYPE = "dataset"
# ROOT_NAME = "ISL_Project"
# PROJECT_NAME = "iSign"
# POSE_FOLDER_NAME = "pose_parts"
# VIDEO_FOLDER_NAME = "video_parts"
#
# POSE_FILES = [
#     "iSign-poses_v1.1_part_aa",
#     "iSign-poses_v1.1_part_ab",
#     "iSign-poses_v1.1_part_ac",
#     "iSign-poses_v1.1_part_ad",
# ]
# VIDEO_FILES = [
#     "iSign-videos_v1.1_part_aa",
#     "iSign-videos_v1.1_part_ab",
# ]
# FILES_TO_TRANSFER = [(name, POSE_FOLDER_NAME) for name in POSE_FILES] + [
#     (name, VIDEO_FOLDER_NAME) for name in VIDEO_FILES
# ]
#
# # Authenticate and initialize services. If already authenticated, Colab reuses the session.
# auth.authenticate_user()
# drive_service = build("drive", "v3")
# hf_api = HfApi()
#
# def find_folder(name, parent_id):
#     q = (
#         f"name = '{name}' and "
#         "mimeType = 'application/vnd.google-apps.folder' and "
#         "trashed = false"
#     )
#     if parent_id:
#         q += f" and '{parent_id}' in parents"
#     result = drive_service.files().list(
#         q=q, fields="files(id,name,parents)", pageSize=100
#     ).execute()
#     matches = result.get("files", [])
#     if matches:
#         return matches[0]["id"]
#     return None
#
# def create_folder(name, parent_id):
#     body = {
#         "name": name,
#         "mimeType": "application/vnd.google-apps.folder",
#         "parents": [parent_id],
#     }
#     result = drive_service.files().create(body=body, fields="id,name").execute()
#     print(f"Created Drive folder: {name}")
#     return result["id"]
#
# def get_or_create_folder(name, parent_id):
#     folder_id = find_folder(name, parent_id)
#     return folder_id if folder_id else create_folder(name, parent_id)
#
# def find_drive_file(name, folder_id):
#     safe_name = name.replace("'", r"\'")
#     q = (
#         f"name = '{safe_name}' and '{folder_id}' in parents "
#         "and trashed = false"
#     )
#     result = drive_service.files().list(
#         q=q, fields="files(id,name,size,mimeType)", pageSize=100
#     ).execute()
#     return result.get("files", [])
#
# def remove_cached_file(filename, downloaded_path=None):
#     """Remove only this file's local HF snapshot link/blob after Drive verification."""
#     cache_root = Path.home() / ".cache/huggingface/hub/datasets--Exploration-Lab--iSign"
#     candidates = set()
#
#     if downloaded_path:
#         p = Path(downloaded_path)
#         if p.exists() or p.is_symlink():
#             candidates.add(p)
#             try:
#                 resolved = p.resolve()
#                 if resolved != p:
#                     candidates.add(resolved)
#             except Exception:
#                 pass
#
#     if cache_root.exists():
#         for p in cache_root.rglob(filename):
#             if p.is_file() or p.is_symlink():
#                 candidates.add(p)
#                 try:
#                     resolved = p.resolve()
#                     if resolved != p and cache_root in resolved.parents:
#                         candidates.add(resolved)
#                 except Exception:
#                     pass
#
#     removed_bytes = 0
#     for p in sorted(candidates, key=lambda x: len(str(x)), reverse=True):
#         try:
#             if p.is_symlink():
#                 p.unlink(missing_ok=True)
#             elif p.is_file():
#                 size = p.stat().st_size
#                 p.unlink(missing_ok=True)
#                 removed_bytes += size
#         except FileNotFoundError:
#             pass
#         except Exception as exc:
#             print(f"Cache cleanup warning for {p}: {exc}")
#     if removed_bytes:
#         print(f"Freed about {removed_bytes / 1e9:.2f} GB from local cache.")
#     return removed_bytes
#
# # Pull authoritative file sizes from the dataset repository metadata.
# print("Reading dataset file metadata from Hugging Face...")
# repo_info = hf_api.dataset_info(REPO_ID, files_metadata=True)
# repo_sizes = {
#     item.rfilename: getattr(item, "size", None)
#     for item in repo_info.siblings
# }
# missing = [name for name, _ in FILES_TO_TRANSFER if name not in repo_sizes]
# if missing:
#     raise RuntimeError(
#         "Could not find these expected files in Hugging Face metadata: "
#         + ", ".join(missing)
#     )
#
# # Resolve folders from My Drive instead of hardcoding folder IDs.
# root_id = find_folder(ROOT_NAME, "root")
# if not root_id:
#     root_id = create_folder(ROOT_NAME, "root")
# project_id = get_or_create_folder(PROJECT_NAME, root_id)
# pose_folder_id = get_or_create_folder(POSE_FOLDER_NAME, project_id)
# video_folder_id = get_or_create_folder(VIDEO_FOLDER_NAME, project_id)
# folder_ids = {
#     POSE_FOLDER_NAME: pose_folder_id,
#     VIDEO_FOLDER_NAME: video_folder_id,
# }
#
# print("\nTransfer plan:")
# total_bytes = sum(int(repo_sizes[name] or 0) for name, _ in FILES_TO_TRANSFER)
# print(f"{len(FILES_TO_TRANSFER)} large files | metadata total ~{total_bytes / 1e9:.2f} GB")
# print("Destination: My Drive / ISL_Project / iSign / [pose_parts or video_parts]")
#
# for index, (filename, folder_name) in enumerate(FILES_TO_TRANSFER, start=1):
#     expected_size = repo_sizes.get(filename)
#     if expected_size is None:
#         raise RuntimeError(f"File size unavailable for {filename}; stopping safely.")
#     expected_size = int(expected_size)
#     folder_id = folder_ids[folder_name]
#
#     print(f"\n{'=' * 72}")
#     print(f"[{index}/{len(FILES_TO_TRANSFER)}] {filename}")
#     print(f"Expected size: {expected_size / 1e9:.2f} GB")
#
#     # Check Drive first. If the exact-size file is already there, don't upload it again.
#     existing_files = find_drive_file(filename, folder_id)
#     matching = next(
#         (f for f in existing_files if int(f.get("size", -1)) == expected_size),
#         None
#     )
#     if matching:
#         print(f"Already complete in Drive; skipping upload. Drive ID: {matching['id']}")
#         # It is safe to clear the local cache copy only after confirming Drive size.
#         remove_cached_file(filename)
#         free_gb = shutil.disk_usage("/").free / 1e9
#         print(f"Local free disk: {free_gb:.2f} GB")
#         continue
#
#     # Remove stale/incomplete same-name Drive entries before uploading a clean copy.
#     for old in existing_files:
#         print(
#             f"Removing mismatched Drive entry ({old.get('size', 'unknown')} bytes) "
#             f"before retry: {old['id']}"
#         )
#         drive_service.files().delete(fileId=old["id"]).execute()
#
#     # If not cached, make sure there's enough disk for this single file plus safety margin.
#     cache_root = Path.home() / ".cache/huggingface/hub/datasets--Exploration-Lab--iSign"
#     cached_candidates = [
#         p for p in cache_root.rglob(filename)
#         if p.is_file() and p.stat().st_size == expected_size
#     ] if cache_root.exists() else []
#
#     downloaded_path = str(cached_candidates[0]) if cached_candidates else None
#     if downloaded_path is None:
#         free_bytes = shutil.disk_usage("/").free
#         safety_margin = 6 * 1024**3
#         required = expected_size + safety_margin
#         if free_bytes < required:
#             raise RuntimeError(
#                 f"Not enough Colab disk space for {filename}. "
#                 f"Free: {free_bytes / 1e9:.2f} GB; "
#                 f"needed about {required / 1e9:.2f} GB. "
#                 "No new download was started. Ensure previous verified cache files "
#                 "were cleared, then rerun this cell."
#             )
#         print("Downloading this file to the local HF cache (one file at a time)...")
#         downloaded_path = hf_hub_download(
#             repo_id=REPO_ID,
#             filename=filename,
#             repo_type=REPO_TYPE,
#         )
#     else:
#         print("Using existing complete cached file; no new download needed.")
#
#     source = Path(downloaded_path)
#     actual_source_size = source.stat().st_size
#     if actual_source_size != expected_size:
#         raise RuntimeError(
#             f"Local file size mismatch for {filename}: "
#             f"{actual_source_size} != expected {expected_size}. "
#             "Keeping the local file for diagnosis."
#         )
#
#     print(f"Local source ready: {source}")
#     print(f"Uploading to Drive folder '{folder_name}'...")
#     media = MediaFileUpload(
#         str(source),
#         mimetype="application/octet-stream",
#         chunksize=8 * 1024 * 1024,
#         resumable=True,
#     )
#     request = drive_service.files().create(
#         body={"name": filename, "parents": [folder_id]},
#         media_body=media,
#         fields="id,name,size",
#     )
#
#     response = None
#     last_reported = -5
#     start_time = time.time()
#     while response is None:
#         status, response = request.next_chunk(num_retries=5)
#         if status:
#             percent = int(status.progress() * 100)
#             if percent >= last_reported + 5 or percent == 100:
#                 elapsed = max(time.time() - start_time, 1)
#                 sent_gb = expected_size * status.progress() / 1e9
#                 speed_mbps = (expected_size * status.progress() / elapsed) / 1e6
#                 print(
#                     f"Upload: {percent:3d}% | {sent_gb:.2f}/{expected_size / 1e9:.2f} GB "
#                     f"| average {speed_mbps:.1f} MB/s"
#                 )
#                 last_reported = percent
#
#     uploaded_id = response["id"]
#     uploaded_size = int(response.get("size", -1))
#
#     # Verify with a fresh Drive metadata lookup, not only the upload response.
#     verified = drive_service.files().get(
#         fileId=uploaded_id, fields="id,name,size,mimeType,parents"
#     ).execute()
#     drive_size = int(verified.get("size", -1))
#     if uploaded_size != expected_size or drive_size != expected_size:
#         raise RuntimeError(
#             f"Drive verification failed for {filename}. "
#             f"Expected {expected_size}; upload response {uploaded_size}; "
#             f"Drive metadata {drive_size}. Local cache has been retained."
#         )
#
#     print(f"VERIFIED: {filename} | {drive_size} bytes | Drive ID: {uploaded_id}")
#
#     # Only delete local cached bytes after exact-size Drive verification.
#     remove_cached_file(filename, downloaded_path)
#     print(f"Finished {filename}. Local free disk: {shutil.disk_usage('/').free / 1e9:.2f} GB")
#
# print("\nALL REQUESTED LARGE FILES ARE VERIFIED IN GOOGLE DRIVE.")
# print("Pose destination:", f"ISL_Project/iSign/{POSE_FOLDER_NAME}")
# print("Video destination:", f"ISL_Project/iSign/{VIDEO_FOLDER_NAME}")
#

## 3. Project code
The five cells below write the project modules to `/content/isl_code`. The same files are in the `isl_ready` folder next to this notebook; the notebook carries copies so it works on its own.

In [ ]:
%%writefile /content/isl_code/isign_pose.py
"""
iSign pose preprocessing, video-level splitting and shard storage.

Numpy-only on purpose (no torch) so that the logic can be tested without a GPU environment.

Frame convention -- do not change silently:
  * 75 landmarks = 33 body + 21 left hand + 21 right hand
    = indices 0:33 and 501:543 of the 576-landmark pose-format layout
  * 225 coordinates per frame, landmark-major: [x0, y0, z0, x1, y1, z1, ...]
  * 1 frame counter appended as column 225  ->  frame width 226
  * counter = frame_index / n_frames   (first frame 0.0, last frame (n - 1) / n)
  * x, y: per sequence, (x - width/2) / (width/2) and (y - height/2) / (height/2) with the image size from the pose
    header (most iSign files are 300 x 300, i.e. (v - 150) / 150; a few are not);  z: kept in source units unless the
    normalization says otherwise
  * invalid landmarks are stored as 0.0 *and* flagged in a separate validity array; the loss uses
    the flags, never the values
"""
import bisect
import hashlib
import io
import json
import math
import os
import shutil
import time
from collections import Counter, OrderedDict
from pathlib import Path

import numpy as np

# ----------------------------------------------------------------------------- representation
N_BODY, N_HAND = 33, 21
N_LANDMARKS = N_BODY + 2 * N_HAND            # 75
N_COORDS = N_LANDMARKS * 3                   # 225
FRAME_WIDTH = N_COORDS + 1                   # 226 (coordinates + counter)
EXPECTED_KEEP_IDX = np.r_[0:33, 501:522, 522:543]   # source landmark indices that are kept

# component slices inside the kept 75-landmark layout
COMPONENTS = OrderedDict([("body", (0, 33)), ("left_hand", (33, 54)), ("right_hand", (54, 75))])
BODY_L_WRIST, BODY_R_WRIST = 15, 16          # MediaPipe pose indices
LH_WRIST, RH_WRIST = 33, 54                  # hand wrists inside the kept layout

DEFAULT_NORMALIZATION = {
    "version": 1,
    "landmarks": "pose-format indices 0:33 (body) + 501:522 (left hand) + 522:543 (right hand)",
    "n_landmarks": N_LANDMARKS,
    "coord_layout": "landmark-major [x0, y0, z0, x1, y1, z1, ...]",
    "frame_width": FRAME_WIDTH,
    "counter": "frame_index / n_frames in the last column (first frame 0.0)",
    # image_size: every pose file carries its own image size in the header (most are 300 x 300, some are not), so
    # x -> (x - width/2) / (width/2) and y -> (y - height/2) / (height/2). For 300 x 300 this is exactly (v - 150) / 150.
    "xy": {"mode": "image_size", "default_size": 300.0,
           "formula": "x: (v - width/2) / (width/2); y: (v - height/2) / (height/2); width and height from each pose header"},
    "z": {"mode": "keep",
          "scale": {"body": 1.0, "left_hand": 1.0, "right_hand": 1.0},
          "formula": "value / scale[component]  (scale is 1.0 in 'keep' mode)"},
    "confidence_threshold": 0.5,
    "zero_fill_is_missing": True,
    "clamp": None,
    "missing_fill": 0.0,
    "storage_dtype": "float16",
}

DEFAULT_QUALITY = {
    "min_frames": 2,
    "max_frames": 600,
    "min_valid_fraction": 0.15,        # share of valid landmarks over the whole sequence
    "max_empty_frame_fraction": 0.5,   # frames with no valid landmark at all
    "require_hands": True,             # at least one valid hand landmark somewhere
}


def make_normalization(z_mode="keep", z_scale=None, confidence_threshold=0.5,
                       zero_fill_is_missing=True, storage_dtype="float16", xy_mode="image_size"):
    params = json.loads(json.dumps(DEFAULT_NORMALIZATION))
    if xy_mode == "constant":          # the earlier assumption: every frame is 300 x 300
        params["xy"] = {"mode": "constant", "center": 150.0, "half_range": 150.0, "formula": "(v - center) / half_range"}
    elif xy_mode != "image_size":
        raise ValueError("xy_mode must be 'image_size' or 'constant'")
    if z_mode not in ("keep", "scale"):
        raise ValueError("z_mode must be 'keep' or 'scale'")
    params["z"]["mode"] = z_mode
    if z_mode == "scale":
        if not z_scale:
            raise ValueError("z_mode='scale' needs z_scale={'body': .., 'left_hand': .., 'right_hand': ..}")
        for name in COMPONENTS:
            params["z"]["scale"][name] = float(z_scale[name])
            if not params["z"]["scale"][name] > 0:
                raise ValueError("z scales must be positive")
    params["confidence_threshold"] = float(confidence_threshold)
    params["zero_fill_is_missing"] = bool(zero_fill_is_missing)
    params["storage_dtype"] = str(np.dtype(storage_dtype))
    return params


def params_hash(params):
    return hashlib.sha1(json.dumps(params, sort_keys=True).encode("utf-8")).hexdigest()[:10]


def header_dims(pose, default=300.0):
    """(width, height) from the pose header; falls back to default x default if missing or not positive."""
    try:
        d = pose.header.dimensions
        w, h = float(d.width), float(d.height)
        if np.isfinite(w) and np.isfinite(h) and w > 0 and h > 0:
            return (w, h)
    except Exception:
        pass
    return (float(default), float(default))


def xy_geometry(params, dims=None):
    """(cx, cy, hx, hy): centre and half range used for x and y of one sequence."""
    xy = params["xy"]
    if xy["mode"] == "constant":
        return xy["center"], xy["center"], xy["half_range"], xy["half_range"]
    w, h = dims if dims is not None else (xy["default_size"], xy["default_size"])
    return w / 2.0, h / 2.0, w / 2.0, h / 2.0


def z_scale_vector(params):
    """Per-landmark z divisor, shape [75]."""
    vec = np.ones(N_LANDMARKS, dtype=np.float32)
    if params["z"]["mode"] == "scale":
        for name, (a, b) in COMPONENTS.items():
            vec[a:b] = params["z"]["scale"][name]
    return vec


# ----------------------------------------------------------------------------- archive access
class VirtualConcatFile(io.BufferedIOBase):
    """Seekable, read-only view of several files joined end to end (nothing is copied or combined)."""

    def __init__(self, paths):
        self.paths = [Path(p) for p in paths]
        self.files = [open(p, "rb") for p in self.paths]
        self.sizes = [p.stat().st_size for p in self.paths]
        self.starts, total = [], 0
        for size in self.sizes:
            self.starts.append(total)
            total += size
        self.total_size = total
        self.pos = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            new_pos = offset
        elif whence == io.SEEK_CUR:
            new_pos = self.pos + offset
        elif whence == io.SEEK_END:
            new_pos = self.total_size + offset
        else:
            raise ValueError("Invalid whence")
        if new_pos < 0:
            raise ValueError("Negative seek position")
        self.pos = new_pos
        return self.pos

    def read(self, size=-1):
        if self.pos >= self.total_size:
            return b""
        if size is None or size < 0:
            size = self.total_size - self.pos
        size = min(size, self.total_size - self.pos)
        chunks = []
        while size > 0:
            idx = bisect.bisect_right(self.starts, self.pos) - 1
            local_pos = self.pos - self.starts[idx]
            take = min(size, self.sizes[idx] - local_pos)
            f = self.files[idx]
            f.seek(local_pos)
            chunk = f.read(take)
            if not chunk:
                break
            chunks.append(chunk)
            self.pos += len(chunk)
            size -= len(chunk)
        return b"".join(chunks)

    def close(self):
        for f in self.files:
            f.close()
        super().close()


# ----------------------------------------------------------------------------- pose -> arrays
def resolve_landmark_layout(pose):
    """
    Return (keep_idx, note). The kept landmarks are fixed (EXPECTED_KEEP_IDX); the file header is only
    used to *verify* that they are the body / left-hand / right-hand components. If the header says
    something different the function raises instead of silently selecting other landmarks.
    """
    note = "header not inspected; using the fixed indices"
    try:
        ranges, offset = {}, 0
        for comp in pose.header.components:
            n = len(comp.points)
            ranges[comp.name] = (offset, offset + n)
            offset += n
        body, left, right = ranges["POSE_LANDMARKS"], ranges["LEFT_HAND_LANDMARKS"], ranges["RIGHT_HAND_LANDMARKS"]
        sizes = (body[1] - body[0], left[1] - left[0], right[1] - right[0])
        total = pose.body.data.shape[2]
        if sizes == (N_BODY, N_HAND, N_HAND) and offset == total:
            idx = np.r_[body[0]:body[1], left[0]:left[1], right[0]:right[1]]
            if not np.array_equal(idx, EXPECTED_KEEP_IDX):
                raise ValueError(
                    "The pose header places body/hands at different indices than the expected "
                    f"0:33 + 501:543 (got body {body}, left {left}, right {right}). Investigate before continuing.")
            return EXPECTED_KEEP_IDX.copy(), f"verified from header: body {body}, left hand {left}, right hand {right}"
        note = f"header sizes {sizes} / total {total} not as expected; using the fixed indices"
    except ValueError:
        raise
    except Exception as exc:  # header missing or unusual
        note = f"header unreadable ({exc!r}); using the fixed indices"
    return EXPECTED_KEEP_IDX.copy(), note


def extract_raw(pose, keep_idx=EXPECTED_KEEP_IDX):
    """
    Raw (un-normalised) values of the kept landmarks for the first person, with every kind of
    'missing' kept apart. Returns None when the pose has no frames.
    """
    data, conf = pose.body.data, pose.body.confidence
    d, dmask = np.ma.getdata(data), np.ma.getmaskarray(data)
    c, cmask = np.ma.getdata(conf), np.ma.getmaskarray(conf)
    if d.ndim != 4 or d.shape[0] == 0 or d.shape[1] == 0:
        return None
    xyz = d[:, 0][:, keep_idx, :].astype(np.float32)           # [T, 75, 3]
    cf = c[:, 0][:, keep_idx].astype(np.float32)               # [T, 75]
    masked = dmask[:, 0][:, keep_idx, :].any(axis=-1) | cmask[:, 0][:, keep_idx]
    nonfinite = ~np.isfinite(xyz).all(axis=-1) | ~np.isfinite(cf)
    zero_filled = (xyz == 0).all(axis=-1)
    xyz = np.nan_to_num(xyz, nan=0.0, posinf=0.0, neginf=0.0)
    cf = np.nan_to_num(cf, nan=0.0, posinf=0.0, neginf=0.0)
    return {"xyz": xyz, "conf": cf, "masked": masked, "nonfinite": nonfinite, "zero_filled": zero_filled,
            "dims": header_dims(pose)}


def validity_categories(raw, params):
    """Boolean arrays [T, 75] for each reason a landmark can be unusable."""
    return {
        "nonfinite": raw["nonfinite"],
        "masked": raw["masked"],
        "low_confidence": raw["conf"] < params["confidence_threshold"],
        "zero_filled": raw["zero_filled"],
    }


def landmark_validity(raw, params):
    """True where a landmark may be used as a target. Rule is explicit and configurable."""
    cats = validity_categories(raw, params)
    invalid = cats["nonfinite"] | cats["masked"] | cats["low_confidence"]
    if params["zero_fill_is_missing"]:
        invalid = invalid | cats["zero_filled"]
    return ~invalid


def normalize_xyz(xyz, params, dims=None):
    """Raw [..., 75, 3] -> normalised. dims = (width, height) of the sequence's image (default: the default size)."""
    out = np.array(xyz, dtype=np.float32, copy=True)
    cx, cy, hx, hy = xy_geometry(params, dims)
    out[..., 0] = (out[..., 0] - cx) / hx
    out[..., 1] = (out[..., 1] - cy) / hy
    out[..., 2] = out[..., 2] / z_scale_vector(params)
    return out


def denormalize_xyz(xyz, params, dims=None):
    """
    Inverse of normalize_xyz (for visualisation / inference output). Accepts [..., 75, 3].
    Generated poses have no image of their own: pass the canvas you want to draw on (default 300 x 300).
    """
    out = np.array(xyz, dtype=np.float32, copy=True)
    cx, cy, hx, hy = xy_geometry(params, dims)
    out[..., 0] = out[..., 0] * hx + cx
    out[..., 1] = out[..., 1] * hy + cy
    out[..., 2] = out[..., 2] * z_scale_vector(params)
    return out


def normalized_xy(raw, params):
    """Normalised x/y [T, 75, 2] of one extract_raw() result, using that sequence's own image size."""
    cx, cy, hx, hy = xy_geometry(params, raw.get("dims"))
    xy = np.array(raw["xyz"][..., :2], dtype=np.float32, copy=True)
    xy[..., 0] = (xy[..., 0] - cx) / hx
    xy[..., 1] = (xy[..., 1] - cy) / hy
    return xy


def raw_to_record(raw, params):
    """Normalise (per-sequence image size), then fill invalid landmarks with params['missing_fill']; keep the flags."""
    valid = landmark_validity(raw, params)
    xyz = normalize_xyz(raw["xyz"], params, raw.get("dims"))
    xyz[~valid] = params["missing_fill"]
    return {"coords": xyz.reshape(len(xyz), N_COORDS), "valid": valid, "dims": raw.get("dims")}


def add_counter(coords):
    """[T, 225] -> [T, 226] with the frame counter frame_index / n_frames in the last column."""
    n = coords.shape[0]
    counter = np.arange(n, dtype=np.float32) / np.float32(n)
    return np.concatenate([np.asarray(coords, dtype=np.float32), counter[:, None]], axis=1)


def quality_reason(record, quality):
    """None if the record is usable, otherwise a short reason for excluding it."""
    coords, valid = record["coords"], record["valid"]
    n = len(coords)
    if n < quality["min_frames"]:
        return f"too short ({n} frames)"
    if n > quality["max_frames"]:
        return f"too long ({n} frames)"
    if not np.isfinite(coords).all():
        return "non-finite values after normalisation"
    if valid.mean() < quality["min_valid_fraction"]:
        return f"too few valid landmarks ({valid.mean():.0%})"
    if (~valid.any(axis=1)).mean() > quality["max_empty_frame_fraction"]:
        return "too many frames without any valid landmark"
    if quality["require_hands"] and not valid[:, COMPONENTS["left_hand"][0]:].any():
        return "no valid hand landmark in any frame"
    return None


# ----------------------------------------------------------------------------- splitting
def video_id_of(uid):
    """iSign UIDs look like '<video_id>-<sequence_number>'."""
    return str(uid).rsplit("-", 1)[0]


def normalize_text(text):
    """Lower-case and collapse whitespace; the stored .text file has exactly one such line per sequence."""
    return " ".join(str(text).split()).lower()


def split_by_video(uids, seed, fractions=(0.8, 0.1, 0.1), names=("train", "val", "test")):
    """
    Assign whole videos to splits so that no video appears in two of them. Videos are shuffled with
    a fixed seed and added greedily until each split reaches its share of *sequences*.
    Returns (uids_by_split, videos_by_split).
    """
    if abs(sum(fractions) - 1.0) > 1e-9 or len(fractions) != len(names):
        raise ValueError("fractions must sum to 1 and match names")
    groups = OrderedDict()
    for u in uids:
        groups.setdefault(video_id_of(u), []).append(u)
    videos = sorted(groups)
    order = np.random.RandomState(seed).permutation(len(videos))
    cuts = np.cumsum(fractions) * len(uids)
    uids_by, videos_by = OrderedDict((n, []) for n in names), OrderedDict((n, []) for n in names)
    seen = 0
    for k in order:
        v = videos[int(k)]
        s = 0
        while s < len(names) - 1 and seen >= cuts[s]:
            s += 1
        uids_by[names[s]].extend(groups[v])
        videos_by[names[s]].append(v)
        seen += len(groups[v])
    return uids_by, videos_by


def split_overlaps(videos_by):
    """Videos (and, separately, nothing else) that occur in more than one split."""
    seen, overlap = {}, set()
    for name, vids in videos_by.items():
        for v in vids:
            if v in seen and seen[v] != name:
                overlap.add(v)
            seen[v] = name
    return sorted(overlap)


def split_report(uids_by, videos_by, fractions, total_matched):
    rows = []
    for (name, uids), frac in zip(uids_by.items(), fractions):
        rows.append({"split": name, "sequences": len(uids), "videos": len(videos_by[name]),
                     "share_of_sequences": len(uids) / max(total_matched, 1), "target_share": frac})
    return rows


# ----------------------------------------------------------------------------- shard storage
def shard_base(out_dir, k):
    return str(Path(out_dir) / f"shard_{k:04d}")


class ShardWriter:
    """Collects converted sequences and writes one shard (text / files / skels / valid / idx / json)."""

    def __init__(self, params):
        self.params = params
        self.dtype = np.dtype(params["storage_dtype"])
        self.uids, self.texts, self.coords, self.valid, self.excluded, self.dims = [], [], [], [], [], []

    def add(self, uid, text, coords, valid, dims=None):
        text = normalize_text(text)
        if not text:
            raise ValueError("empty text")
        self.uids.append(str(uid))
        self.texts.append(text)
        self.coords.append(np.asarray(coords, dtype=np.float32))
        self.valid.append(np.asarray(valid, dtype=bool))
        self.dims.append([float(dims[0]), float(dims[1])] if dims is not None else None)   # image size used for x/y

    def exclude(self, uid, reason):
        self.excluded.append({"uid": str(uid), "reason": str(reason)})

    def __len__(self):
        return len(self.uids)

    def write(self, out_dir, k, manifest_start, manifest_end, row_uids=None):
        Path(out_dir).mkdir(parents=True, exist_ok=True)
        base = shard_base(out_dir, k)
        lengths = [len(c) for c in self.coords]
        offsets = np.concatenate([[0], np.cumsum(lengths)]).astype(np.int64)
        all_c = np.concatenate(self.coords) if self.coords else np.zeros((0, N_COORDS), np.float32)
        all_v = np.concatenate(self.valid) if self.valid else np.zeros((0, N_LANDMARKS), bool)
        stored = all_c.astype(self.dtype)
        # quantisation error: absolute for |value| < 1, relative otherwise
        qerr = float((np.abs(stored.astype(np.float32) - all_c) / np.maximum(np.abs(all_c), 1.0)).max()) if all_c.size else 0.0

        def atomic(ext, writer):
            tmp = f"{base}.{ext}.tmp"
            try:
                writer(tmp)
                os.replace(tmp, f"{base}.{ext}")
            except BaseException:      # e.g. disk full: do not leave half-written files behind
                try:
                    os.remove(tmp)
                except OSError:
                    pass
                raise

        def write_text(path, text):
            with open(path, "w", encoding="utf-8", newline="\n") as f:
                f.write(text)

        def write_idx(path):
            with open(path, "wb") as f:
                np.save(f, offsets)

        atomic("text", lambda p: write_text(p, "".join(t + "\n" for t in self.texts)))
        atomic("files", lambda p: write_text(p, "".join(u + "\n" for u in self.uids)))
        atomic("skels", lambda p: stored.tofile(p))
        atomic("valid", lambda p: all_v.astype(np.uint8).tofile(p))
        atomic("idx.npy", write_idx)
        meta = {
            "shard": int(k), "n_sequences": len(self.uids), "n_frames": int(offsets[-1]),
            "dtype": str(self.dtype), "manifest_start": int(manifest_start), "manifest_end": int(manifest_end),
            "max_quant_error": qerr, "normalization_hash": params_hash(self.params),
            "lengths": [int(x) for x in lengths],
            "valid_fraction": [float(v.mean()) for v in self.valid],
            "dims": self.dims,
            "excluded": self.excluded,
            "row_first_uid": row_uids[0] if row_uids else None,      # identity of the manifest rows this shard covers
            "row_last_uid": row_uids[1] if row_uids else None,
        }
        atomic("json", lambda p: write_text(p, json.dumps(meta)))   # written last: marks the shard complete
        return meta


def convert_manifest(rows, out_dir, read_pose, params, quality, shard_size,
                     max_shards=None, keep_idx=EXPECTED_KEEP_IDX, log=print,
                     min_free_gb=None, disk_path="/"):
    """
    Convert (uid, text) rows into shards. Resumable: shards whose .json exists are skipped, after checking
    that they were made from the same rows in the same order (a different order raises).
    Every excluded sequence is recorded with a reason in its shard's metadata.
    read_pose(uid) must return a pose object (or raise).
    If min_free_gb is given, conversion stops cleanly (stopped='low disk') before a shard when the free
    space on disk_path is below it; rerun later to continue.
    """
    n_shards = math.ceil(len(rows) / shard_size) if rows else 0
    todo = n_shards if max_shards is None else min(int(max_shards), n_shards)
    done = skipped_existing = 0
    stopped = None
    for k in range(todo):
        a, b = k * shard_size, min(len(rows), (k + 1) * shard_size)
        meta_path = Path(shard_base(out_dir, k) + ".json")
        if meta_path.exists():
            old = json.loads(meta_path.read_text(encoding="utf-8"))
            if old.get("row_first_uid") != rows[a][0] or old.get("row_last_uid") != rows[b - 1][0]:
                raise ValueError(f"{meta_path.name} was made from different rows (or a different row order) than the ones "
                                 "given now; use a new output folder or delete the old shards")
            skipped_existing += 1
            continue
        if min_free_gb is not None:
            free_gb = shutil.disk_usage(disk_path).free / 1e9
            if free_gb < min_free_gb:
                stopped = f"low disk ({free_gb:.1f} GB free on {disk_path}, limit {min_free_gb} GB)"
                log(f"  stopping before shard {k:04d}: {stopped}. Free some space and rerun; finished shards are kept.")
                break
        started = time.time()
        writer = ShardWriter(params)
        for uid, text in rows[a:b]:
            try:
                pose = read_pose(uid)
            except Exception as exc:   # corrupt archive entry, missing file, ...
                writer.exclude(uid, f"unreadable pose ({type(exc).__name__}: {str(exc)[:80]})")
                continue
            if pose is None:
                writer.exclude(uid, "no pose entry")
                continue
            try:
                raw = extract_raw(pose, keep_idx)
                if raw is None:
                    writer.exclude(uid, "pose has no frames")
                    continue
                record = raw_to_record(raw, params)
            except Exception as exc:
                writer.exclude(uid, f"pose structure error ({type(exc).__name__}: {str(exc)[:80]})")
                continue
            reason = quality_reason(record, quality)
            if reason:
                writer.exclude(uid, reason)
                continue
            writer.add(uid, text, record["coords"], record["valid"], dims=record.get("dims"))
        meta = writer.write(out_dir, k, a, b, row_uids=(rows[a][0], rows[b - 1][0]))
        done += 1
        secs = time.time() - started
        log(f"  shard {k:04d}: kept {meta['n_sequences']}/{b - a} sequences, {meta['n_frames']} frames, "
            f"quantisation error {meta['max_quant_error']:.1e}, {secs:.0f}s ({secs / max(b - a, 1):.2f}s per sequence)")
    return {"shards_total": n_shards, "converted_now": done, "already_present": skipped_existing, "stopped": stopped}


class ShardStore:
    """Read access to the shards of one split (memory-mapped, opened lazily, safe to fork)."""

    def __init__(self, shard_dir, expected_hash=None, max_shards=None):
        self.dir = Path(shard_dir)
        self.metas = []
        k = 0
        while max_shards is None or k < max_shards:
            meta_path = Path(shard_base(self.dir, k) + ".json")
            if not meta_path.exists():
                break
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
            if expected_hash is not None and meta["normalization_hash"] != expected_hash:
                raise ValueError(f"{meta_path.name}: normalization hash {meta['normalization_hash']} "
                                 f"differs from the expected {expected_hash}")
            self.metas.append(meta)
            k += 1
        self.uids, self.texts, self.locs = [], [], []
        lengths, vfrac, dims = [], [], []
        for pos, meta in enumerate(self.metas):
            base = shard_base(self.dir, meta["shard"])
            files = Path(base + ".files").read_text(encoding="utf-8").splitlines()
            texts = Path(base + ".text").read_text(encoding="utf-8").splitlines()
            if not (len(files) == len(texts) == meta["n_sequences"] == len(meta["lengths"])):
                raise ValueError(f"shard {meta['shard']}: files/text/meta counts differ "
                                 f"({len(files)}/{len(texts)}/{meta['n_sequences']})")
            self.uids += files
            self.texts += texts
            self.locs += [(pos, i) for i in range(len(files))]
            lengths += meta["lengths"]
            vfrac += meta["valid_fraction"]
            dims += meta.get("dims") or [None] * len(files)
        self.lengths = np.asarray(lengths, dtype=np.int64)
        self.valid_fraction = np.asarray(vfrac, dtype=np.float64)
        self.dims = [tuple(d) if d is not None else None for d in dims]       # (width, height) used for each sequence
        self._open = {}

    def __len__(self):
        return len(self.uids)

    def __getstate__(self):
        """Pickle without the open memory maps (they would be copied into every spawned worker)."""
        state = self.__dict__.copy()
        state["_open"] = {}
        return state

    def exclusions(self):
        return [e for m in self.metas for e in m["excluded"]]

    def close(self):
        """Release the memory maps (needed before deleting the files on Windows)."""
        for coords, valid, _ in self._open.values():
            for arr in (coords, valid):
                mm = getattr(arr, "_mmap", None)
                if mm is not None:
                    mm.close()
        self._open.clear()

    def _arrays(self, pos):
        if pos not in self._open:
            meta = self.metas[pos]
            base = shard_base(self.dir, meta["shard"])
            n = meta["n_frames"]
            coords = np.memmap(base + ".skels", dtype=np.dtype(meta["dtype"]), mode="r", shape=(n, N_COORDS))
            valid = np.memmap(base + ".valid", dtype=np.uint8, mode="r", shape=(n, N_LANDMARKS))
            offsets = np.load(base + ".idx.npy")
            self._open[pos] = (coords, valid, offsets)
        return self._open[pos]

    def get(self, i):
        """(uid, text, coords float32 [T, 225], valid bool [T, 75])"""
        pos, j = self.locs[i]
        coords, valid, offsets = self._arrays(pos)
        a, b = int(offsets[j]), int(offsets[j + 1])
        return (self.uids[i], self.texts[i],
                np.asarray(coords[a:b], dtype=np.float32), np.asarray(valid[a:b]).astype(bool))


# ----------------------------------------------------------------------------- diagnostics
def _pcts(a, qs=(0, 1, 5, 25, 50, 75, 95, 99, 100)):
    a = np.asarray(a)
    if a.size == 0:
        return {f"p{q}": None for q in qs}
    return {f"p{q}": float(v) for q, v in zip(qs, np.percentile(a, qs))}


def _iqr(a):
    a = np.asarray(a)
    return float(np.subtract(*np.percentile(a, [75, 25]))) if a.size else None


def _pair_distance(raws, valid_list, i, j, params):
    """Distance between two landmarks in NORMALISED x/y units (each sequence with its own image size)."""
    out = []
    for r, v in zip(raws, valid_list):
        m = v[:, i] & v[:, j]
        if m.any():
            xy = normalized_xy(r, params)[m]
            out.append(np.linalg.norm(xy[:, i, :] - xy[:, j, :], axis=1))
    return np.concatenate(out) if out else np.zeros(0)


LANDMARK_GROUPS = OrderedDict([
    ("face 0-10", range(0, 11)), ("shoulders 11-12", range(11, 13)), ("elbows 13-14", range(13, 15)),
    ("wrists 15-16", range(15, 17)), ("pose hand points 17-22", range(17, 23)), ("hips 23-24", range(23, 25)),
    ("knees 25-26", range(25, 27)), ("ankles/feet 27-32", range(27, 33)), ("left hand", range(33, 54)),
    ("right hand", range(54, 75))])


def estimate_z_scales(raws, params):
    """Per-component z scale from a TRAINING-split sample: 99th percentile of |z| over valid landmarks."""
    valid_list = [landmark_validity(r, params) for r in raws]
    scales = {}
    for name, (a, b) in COMPONENTS.items():
        z = np.concatenate([np.abs(r["xyz"][:, a:b, 2][v[:, a:b]]) for r, v in zip(raws, valid_list)])
        scales[name] = float(max(np.percentile(z, 99), 1e-6)) if z.size else 1.0
    return scales


def diagnose_sample(raws, params, jump_threshold=0.5, z_ratio_range=(0.1, 10.0),
                    out_of_bounds_warn=2.0, out_of_bounds_fail=20.0):
    """
    Distribution / validity / coherence report over a list of extract_raw() results.
    Returns (report, flags) where flags is a list of (severity, message); severity is 'WARN' or 'FAIL'.
    """
    zvec = z_scale_vector(params)
    geoms = [xy_geometry(params, r.get("dims")) for r in raws]          # (cx, cy, hx, hy) of each sequence
    nxy = [normalized_xy(r, params) for r in raws]                      # normalised x/y with each sequence's own image size
    valid_list = [landmark_validity(r, params) for r in raws]
    flags = []

    # ---- landmark-level breakdown (categories overlap; 'valid' is what remains)
    thr = params["confidence_threshold"]
    total = sum(v.size for v in valid_list)
    breakdown = {
        "total_landmarks": int(total),
        "valid": int(sum(v.sum() for v in valid_list)),
        "nonfinite": int(sum(r["nonfinite"].sum() for r in raws)),
        "masked": int(sum(r["masked"].sum() for r in raws)),
        "low_confidence": int(sum((r["conf"] < thr).sum() for r in raws)),
        "zero_filled": int(sum(r["zero_filled"].sum() for r in raws)),
        "zero_filled_but_confident": int(sum((r["zero_filled"] & (r["conf"] >= thr)).sum() for r in raws)),
        "confident_and_not_zero_filled": int(sum(((r["conf"] >= thr) & ~r["zero_filled"]).sum() for r in raws)),
    }
    breakdown["valid_fraction"] = breakdown["valid"] / max(total, 1)
    all_conf = np.concatenate([r["conf"].ravel() for r in raws]) if raws else np.zeros(0)
    conf_report = {"min": float(all_conf.min()) if all_conf.size else None,
                   "max": float(all_conf.max()) if all_conf.size else None, **_pcts(all_conf, (5, 25, 50, 75, 95))}

    if breakdown["nonfinite"]:
        flags.append(("WARN", f"{breakdown['nonfinite']} landmarks had NaN/inf values (treated as invalid)"))
    if breakdown["zero_filled_but_confident"]:
        flags.append(("WARN", f"{breakdown['zero_filled_but_confident']} landmarks are exactly (0,0,0) but have confidence "
                              f">= {thr}; they are treated as missing because zero_fill_is_missing={params['zero_fill_is_missing']}"))
    if conf_report["min"] is not None and (conf_report["min"] < 0 or conf_report["max"] > 1.0 + 1e-6):
        flags.append(("FAIL", f"confidence outside [0, 1]: {conf_report['min']:.3f}..{conf_report['max']:.3f}"))
    if breakdown["valid_fraction"] < 0.2:
        flags.append(("FAIL", f"only {breakdown['valid_fraction']:.0%} of landmarks are valid"))

    # ---- per component
    components = {}
    for name, (a, b) in COMPONENTS.items():
        xs, ys, zs, xns, yns, outs = [], [], [], [], [], []
        for r, v, n in zip(raws, valid_list, nxy):
            m = v[:, a:b]
            xyz = r["xyz"][:, a:b, :]
            xs.append(xyz[..., 0][m])
            ys.append(xyz[..., 1][m])
            zs.append(xyz[..., 2][m])
            nx, ny = n[:, a:b, 0][m], n[:, a:b, 1][m]
            xns.append(nx)
            yns.append(ny)
            outs.append((np.abs(nx) > 1) | (np.abs(ny) > 1))             # beyond this sequence's own image
        x, y, z, xn, yn, out_flags = (np.concatenate(t) if t else np.zeros(0) for t in (xs, ys, zs, xns, yns, outs))
        zn = z / float(zvec[a])
        outside = out_flags.mean() * 100 if x.size else None
        ratio = (_iqr(zn) / _iqr(xn)) if x.size and _iqr(xn) else None
        components[name] = {
            "n_valid_landmarks": int(x.size),
            "x_raw": _pcts(x), "y_raw": _pcts(y), "z_raw": _pcts(z),
            "x_norm": _pcts(xn), "y_norm": _pcts(yn), "z_effective": _pcts(zn),
            "pct_xy_outside_nominal_bounds": None if outside is None else float(outside),
            "z_iqr_over_x_iqr": None if ratio is None else float(ratio),
        }
        if outside is not None:
            if outside > out_of_bounds_fail:
                flags.append(("FAIL", f"{name}: {outside:.1f}% of valid x/y lie outside their image; "
                                      "the x/y normalisation assumption is wrong"))
            elif outside > out_of_bounds_warn:
                flags.append(("WARN", f"{name}: {outside:.1f}% of valid x/y lie outside their image "
                                      "(landmarks the pose estimator placed beyond the frame; kept as is, not clamped)"))
        if ratio is not None and ratio > z_ratio_range[1]:
            flags.append(("FAIL" if params["z"]["mode"] == "keep" else "WARN",
                          f"{name}: z spread is {ratio:.3g}x the x spread (limit {z_ratio_range[1]}): z would dominate a "
                          f"squared-error loss; z mode is '{params['z']['mode']}'"))
        elif ratio is not None and ratio < z_ratio_range[0]:
            flags.append(("WARN", f"{name}: z spread is only {ratio:.3g}x the x spread (below {z_ratio_range[0]}): z carries little "
                                  "weight in the loss; that is a modelling choice, not a corrupted scale"))

    # ---- hands vs body wrists: same coordinate system and consistent left/right?
    pairs = {"LL": (BODY_L_WRIST, LH_WRIST), "RR": (BODY_R_WRIST, RH_WRIST),
             "LR": (BODY_L_WRIST, RH_WRIST), "RL": (BODY_R_WRIST, LH_WRIST)}
    wrist = {}
    for key, (i, j) in pairs.items():
        d = _pair_distance(raws, valid_list, i, j, params)
        wrist[key] = {"n": int(d.size), "median_norm": float(np.median(d)) if d.size else None, **_pcts(d, (25, 75, 95))}
    if all(wrist[k]["median_norm"] is not None for k in pairs):
        if not (wrist["LL"]["median_norm"] < wrist["LR"]["median_norm"] and wrist["RR"]["median_norm"] < wrist["RL"]["median_norm"]):
            flags.append(("FAIL", "hand wrists sit closer to the OPPOSITE body wrist than to their own: left/right ordering or "
                                  f"coordinate systems are inconsistent ({ {k: round(v['median_norm'], 3) for k, v in wrist.items()} })"))
        for key in ("LL", "RR"):
            if wrist[key]["median_norm"] > 0.25:
                flags.append(("WARN", f"{key}: hand wrist and body wrist are a median {wrist[key]['median_norm']:.2f} "
                                      "(normalised units) apart; the components may not share a coordinate system"))
    else:
        flags.append(("WARN", "not enough frames with both a valid hand wrist and body wrist to compare coordinate systems"))
    hand_z = {}
    for key, idx in (("left_wrist_z_abs_median", LH_WRIST), ("right_wrist_z_abs_median", RH_WRIST)):
        vals = np.concatenate([np.abs(r["xyz"][:, idx, 2][v[:, idx]]) for r, v in zip(raws, valid_list)]) if raws else np.zeros(0)
        hand_z[key] = float(np.median(vals)) if vals.size else None

    # ---- temporal behaviour
    jumps, seq_with_jump, empty_fracs = [], 0, []
    for r, v, xy in zip(raws, valid_list, nxy):
        both = v[1:] & v[:-1]
        if both.any():
            d = np.linalg.norm(xy[1:] - xy[:-1], axis=-1)[both]
            jumps.append(d)
            seq_with_jump += int((d > jump_threshold).any())
        empty_fracs.append(float((~v.any(axis=1)).mean()))
    jumps = np.concatenate(jumps) if jumps else np.zeros(0)
    temporal = {
        "frame_to_frame_xy_displacement": _pcts(jumps, (50, 90, 99, 99.9, 100)),
        "pct_displacements_over_threshold": float((jumps > jump_threshold).mean() * 100) if jumps.size else None,
        "jump_threshold_norm": jump_threshold,
        "sequences_with_a_jump": int(seq_with_jump),
        "frames_without_any_valid_landmark_pct": float(np.mean(empty_fracs) * 100) if empty_fracs else None,
    }
    if temporal["pct_displacements_over_threshold"] and temporal["pct_displacements_over_threshold"] > 1.0:
        flags.append(("WARN", f"{temporal['pct_displacements_over_threshold']:.2f}% of frame-to-frame displacements exceed "
                              f"{jump_threshold} (normalised units): implausible discontinuities"))

    # ---- image sizes seen, and which landmark groups carry the target variance
    size_counts = Counter(r.get("dims") for r in raws)
    image_sizes = [{"width": None if s is None else s[0], "height": None if s is None else s[1], "sequences": n}
                   for s, n in size_counts.most_common()]
    per_var, per_rate, per_out = np.zeros(N_LANDMARKS), np.zeros(N_LANDMARKS), np.zeros(N_LANDMARKS)
    for j in range(N_LANDMARKS):
        sel = [v[:, j] for v in valid_list]
        xv = np.concatenate([n[:, j, 0][s] for n, s in zip(nxy, sel)]) if raws else np.zeros(0)
        yv = np.concatenate([n[:, j, 1][s] for n, s in zip(nxy, sel)]) if raws else np.zeros(0)
        zv = np.concatenate([r["xyz"][:, j, 2][s] / zvec[j] for r, s in zip(raws, sel)]) if raws else np.zeros(0)
        per_rate[j] = float(np.concatenate(sel).mean()) if raws else 0.0
        if xv.size > 1:
            per_out[j] = 100.0 * float(((np.abs(xv) > 1) | (np.abs(yv) > 1)).mean())
            per_var[j] = float(xv.var() + yv.var() + zv.var()) * per_rate[j]
    total_var = per_var.sum() or 1.0
    landmark_groups = [{"group": g, "landmarks": len(idx), "valid_pct": 100 * float(per_rate[list(idx)].mean()),
                        "outside_image_pct": float(per_out[list(idx)].mean()),
                        "variance_share_pct": 100 * float(per_var[list(idx)].sum() / total_var)}
                       for g, idx in LANDMARK_GROUPS.items()]

    report = {
        "n_sequences": len(raws), "n_frames": int(sum(len(r["xyz"]) for r in raws)),
        "image_sizes": image_sizes, "landmark_groups": landmark_groups,
        "landmark_breakdown": breakdown, "confidence": conf_report,
        "components": components, "wrist_pairing": wrist, "hand_wrist_z": hand_z, "temporal": temporal,
        "normalization_hash": params_hash(params),
        "flags": [{"severity": s, "message": m} for s, m in flags],
    }
    return report, flags


# ----------------------------------------------------------------------------- plots
POSE_EDGES = [(0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8), (9, 10), (11, 12), (11, 13), (13, 15),
              (15, 17), (15, 19), (15, 21), (17, 19), (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
              (11, 23), (12, 24), (23, 24), (23, 25), (24, 26), (25, 27), (26, 28), (27, 29), (28, 30), (29, 31),
              (30, 32), (27, 31), (28, 32)]
HAND_EDGES = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11), (11, 12),
              (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]


def all_edges():
    """Edges in the kept 75-landmark layout."""
    e = [(a, b) for a, b in POSE_EDGES]
    e += [(33 + a, 33 + b) for a, b in HAND_EDGES]
    e += [(54 + a, 54 + b) for a, b in HAND_EDGES]
    return e


def plot_distributions(raws, params, path):
    import matplotlib
    matplotlib.use("Agg", force=False)
    import matplotlib.pyplot as plt
    xy_p = params["xy"]
    edges_xy = (xy_p["center"] - xy_p["half_range"], xy_p["center"] + xy_p["half_range"]) if xy_p["mode"] == "constant" \
        else (0.0, xy_p["default_size"])          # the common 300 x 300 frame; other sizes are listed in the report
    valid_list = [landmark_validity(r, params) for r in raws]
    fig, axes = plt.subplots(3, 3, figsize=(13, 9))
    for row, (name, (a, b)) in enumerate(COMPONENTS.items()):
        for col, axis in enumerate("xyz"):
            vals = np.concatenate([r["xyz"][:, a:b, col][v[:, a:b]] for r, v in zip(raws, valid_list)])
            ax = axes[row, col]
            if vals.size:
                ax.hist(vals, bins=100, color="#3a6ea5")
            if axis in "xy":
                for edge in edges_xy:
                    ax.axvline(edge, color="crimson", linestyle="--", linewidth=1)
            ax.set_title(f"{name} {axis} (raw, valid only)", fontsize=9)
    fig.suptitle("Raw coordinate distributions (red dashes = bounds of the common 300 x 300 frame)")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


def plot_skeletons(raws, params, path, n_sequences=3, frames_per_sequence=4):
    import matplotlib
    matplotlib.use("Agg", force=False)
    import matplotlib.pyplot as plt
    edges = all_edges()
    chosen = [r for r in raws if len(r["xyz"]) >= frames_per_sequence][:n_sequences]
    fig, axes = plt.subplots(len(chosen), frames_per_sequence, figsize=(3.2 * frames_per_sequence, 3.4 * len(chosen)), squeeze=False)
    colors = {"body": "#222222", "left_hand": "#d1495b", "right_hand": "#2e86ab"}
    for row, r in enumerate(chosen):
        v = landmark_validity(r, params)
        xy = normalized_xy(r, params)
        for col, t in enumerate(np.linspace(0, len(xy) - 1, frames_per_sequence).astype(int)):
            ax = axes[row, col]
            for a, b in edges:
                if v[t, a] and v[t, b]:
                    comp = "body" if a < 33 else ("left_hand" if a < 54 else "right_hand")
                    ax.plot([xy[t, a, 0], xy[t, b, 0]], [xy[t, a, 1], xy[t, b, 1]], color=colors[comp], linewidth=1)
            for name, (a, b) in COMPONENTS.items():
                m = v[t, a:b]
                ax.scatter(xy[t, a:b, 0][m], xy[t, a:b, 1][m], s=6, color=colors[name])
            ax.set_xlim(-1.2, 1.2)
            ax.set_ylim(1.2, -1.2)
            ax.set_aspect("equal")
            ax.set_title(f"seq {row} frame {t}", fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle("Skeleton reconstruction (normalised x/y, valid landmarks only; y axis points down)")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


def plot_wrist_tracks(raws, params, path, index=0):
    import matplotlib
    matplotlib.use("Agg", force=False)
    import matplotlib.pyplot as plt
    r = raws[index]
    v = landmark_validity(r, params)
    xy = normalized_xy(r, params)
    t = np.arange(len(xy))
    fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
    for col, (label, body_i, hand_i) in enumerate((("left", BODY_L_WRIST, LH_WRIST), ("right", BODY_R_WRIST, RH_WRIST))):
        for row, axis in enumerate("xy"):
            ax = axes[row, col]
            for idx, name, style in ((body_i, "body wrist", "-"), (hand_i, "hand wrist", "--")):
                series = np.where(v[:, idx], xy[:, idx, row], np.nan)
                ax.plot(t, series, style, label=name)
            ax.set_title(f"{label} wrist, {axis} (normalised)", fontsize=9)
            ax.legend(fontsize=7)
    fig.suptitle("Body wrist vs hand wrist over time (should move together)")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)

In [ ]:
%%writefile /content/isl_code/isign_report.py
"""
Test registry and final readiness decision. Pure Python (no numpy / torch).

A check only counts as PASS when its block ran to the end without an exception. A block that
cannot be verified in the current environment (for example CUDA tests on a CPU runtime) must call
`chk.unverified("reason")`, which is recorded as UNVERIFIED and blocks the READY decision.
"""
import json
import traceback
from collections import OrderedDict

PASS, FAIL, UNVERIFIED = "PASS", "FAIL", "UNVERIFIED"


class Checks:
    def __init__(self):
        self.results = OrderedDict()

    def record(self, name, status, detail=""):
        self.results[name] = {"status": status, "detail": str(detail)}

    def status(self, name):
        return self.results.get(name, {}).get("status", UNVERIFIED)

    def all_passed(self, names):
        return all(self.status(n) == PASS for n in names)

    def __call__(self, name):
        return _Check(self, name)

    def table(self, names=None):
        names = list(self.results) if names is None else names
        rows = []
        for n in names:
            r = self.results.get(n, {"status": UNVERIFIED, "detail": "not run"})
            rows.append((r["status"], n, r["detail"]))
        return rows

    def print_table(self, names=None):
        for status, name, detail in self.table(names):
            print(f"[{status:10s}] {name}" + (f"  --  {detail}" if detail else ""))


class _Check:
    def __init__(self, registry, name):
        self.registry, self.name = registry, name
        self.detail = ""
        self._unverified = None

    def unverified(self, reason):
        self._unverified = reason

    def note(self, text):
        self.detail = text

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            if self._unverified:
                self.registry.record(self.name, UNVERIFIED, self._unverified)
            else:
                self.registry.record(self.name, PASS, self.detail)
        elif issubclass(exc_type, AssertionError):
            self.registry.record(self.name, FAIL, str(exc) or "assertion failed")
            print(f"[FAIL] {self.name}: {exc}")
        else:
            self.registry.record(self.name, FAIL, f"{exc_type.__name__}: {exc}")
            print(f"[FAIL] {self.name}: {exc_type.__name__}: {exc}")
            traceback.print_exception(exc_type, exc, tb)
        return True   # keep the notebook going; the final table shows every failure


def decide(checks, required):
    """READY only if every required check is PASS."""
    blockers = []
    for name in required:
        r = checks.results.get(name, {"status": UNVERIFIED, "detail": "not run"})
        if r["status"] != PASS:
            blockers.append({"check": name, "status": r["status"], "detail": r["detail"]})
    return ("READY FOR CONTROLLED TRAINING" if not blockers else "NOT READY"), blockers


def render_markdown(title, decision, blockers, checks, sections):
    lines = [f"# {title}", "", f"## Decision: {decision}", ""]
    if blockers:
        lines += ["Blockers:", ""]
        lines += [f"* **{b['check']}** is {b['status']}" + (f": {b['detail']}" if b["detail"] else "") for b in blockers]
        lines.append("")
    lines += ["## Test report", "", "| Status | Check | Detail |", "|---|---|---|"]
    for status, name, detail in checks.table():
        lines.append(f"| {status} | {name} | {str(detail).replace('|', '/')} |")
    for heading, body in sections.items():
        lines += ["", f"## {heading}", "", body if isinstance(body, str) else "```json\n" + json.dumps(body, indent=2) + "\n```"]
    return "\n".join(lines) + "\n"

In [ ]:
%%writefile /content/isl_code/repo_patches.py
"""
Targeted, asserted, idempotent patches for the ProgressiveTransformersSLP repository
(verified against commit ff8b28cb0817d177f63dfa18c7c322806ee5dd96).

Every edit checks that the code it expects is really there and stops with a clear message if the
repository has changed. Original files are copied to <repo>/_originals before the first edit.
"""
import re
import shutil
from pathlib import Path

MARK = "# [isign-patch]"
TESTED_COMMIT = "ff8b28cb0817d177f63dfa18c7c322806ee5dd96"


def _replace_block(src, start_marker, end_marker, new_lines, what):
    """Replace the lines from the one containing start_marker through the one containing end_marker."""
    lines = src.split("\n")
    starts = [i for i, l in enumerate(lines) if start_marker in l]
    ends = [i for i, l in enumerate(lines) if end_marker in l]
    assert len(starts) == 1 and len(ends) == 1, \
        f"{what}: expected exactly one start and one end line, found {len(starts)}/{len(ends)}"
    s, e = starts[0], ends[0]
    assert 0 <= e - s <= 12, f"{what}: unexpected block size ({e - s + 1} lines)"
    return "\n".join(lines[:s] + new_lines + lines[e + 1:])


def _once(src, old, what):
    assert src.count(old) == 1, f"{what}: expected exactly one occurrence, found {src.count(old)}"


# ------------------------------------------------------------------------------------ vocabulary.py
def patch_vocabulary(s, ctx):
    s2 = re.sub(r"^from torchtext[^\n]*\n", "", s, flags=re.M)
    assert s2 != s, "no torchtext import found"
    assert "dataset: Dataset" in s2, "build_vocab signature not found"
    s2 = s2.replace("dataset: Dataset", "dataset")
    new_lines = [
        f"        known = set(self.itos)  {MARK} O(1) membership test instead of scanning the list",
        "        for t in tokens:",
        "            new_index = len(self.itos)",
        "            if t not in known:",
        "                known.add(t)",
        "                self.itos.append(t)",
        "                self.stoi[t] = new_index",
    ]
    return _replace_block(s2, "for t in tokens:", "self.stoi[t] = new_index", new_lines, "vocabulary.add_tokens")


# ------------------------------------------------------------------------------------ helpers.py
def patch_helpers(s, ctx):
    s2 = re.sub(r"^from torchtext[^\n]*\n", "", s, flags=re.M)
    assert s2 != s, "no torchtext import found"
    old = "from torch.utils.tensorboard import SummaryWriter\n"
    _once(s2, old, "tensorboard import")
    new = ("try:\n"
           "    from torch.utils.tensorboard import SummaryWriter\n"
           "except Exception:  # only needed for training logs\n"
           "    SummaryWriter = None\n")
    return s2.replace(old, new)


# ------------------------------------------------------------------------------------ transformer_layers.py
def patch_transformer_layers(s, ctx):
    old = "max_len: int = 200000"
    _once(s, old, "positional-encoding max_len")
    return s.replace(old, f"max_len: int = {ctx['pe_max_len']}")


# ------------------------------------------------------------------------------------ batch.py
def patch_batch(s, ctx):
    old = "        self.trg_mask = None\n"
    _once(s, old, "trg_mask initialisation")
    s = s.replace(old, old + f"        self.frame_mask = None  {MARK}\n"
                            "        self.coord_mask = None\n"
                            "        self.frame_lengths = None\n")

    new_block = [
        f"            {MARK} masks come from true lengths / validity flags, never from coordinate values",
        "            seq_len = self.trg_input.shape[1]",
        "            self.frame_lengths = getattr(torch_batch, \"frame_lengths\", None)",
        "            if self.frame_lengths is not None:",
        "                input_lengths = (self.frame_lengths.cpu() - 1).clamp_min(0)",
        "                valid_steps = torch.arange(seq_len).unsqueeze(0) < input_lengths.unsqueeze(1)",
        "            else:",
        "                valid_steps = torch.ones((self.trg_input.shape[0], seq_len), dtype=torch.bool)",
        "            self.frame_mask = valid_steps  # [batch, T]: True for real (non-padded) target frames",
        "            # decoder padding mask [batch, 1, T, T]: True where the key frame is real",
        "            self.trg_mask = valid_steps.unsqueeze(1).unsqueeze(2).expand(-1, 1, seq_len, -1).contiguous()",
        "            landmark_valid = getattr(torch_batch, \"landmark_valid\", None)",
        "            if landmark_valid is not None:",
        "                if self.future_prediction != 0:",
        "                    raise NotImplementedError(\"landmark validity masks are not supported with future_prediction\")",
        "                # validity of the frames being predicted (shifted by one like self.trg): [B, T, 75] -> [B, T, 225]",
        "                lv = landmark_valid[:, 1:, :]",
        "                self.coord_mask = lv.unsqueeze(-1).expand(-1, -1, -1, 3).reshape(lv.shape[0], lv.shape[1], -1)",
        "                assert self.coord_mask.shape[1] == self.trg.shape[1], \"validity mask and target length differ\"",
    ]
    s = _replace_block(s, "Target Pad is dynamic", "self.trg_mask = (F.pad(", new_block, "batch.py mask block")

    old = "        self.src_mask = self.src_mask.cuda()\n"
    _once(s, old, "_make_cuda src_mask line")
    s = s.replace(old, old + "        self.src_lengths = self.src_lengths.cuda()\n")

    old = "            self.trg_mask = self.trg_mask.cuda()\n"
    _once(s, old, "_make_cuda trg_mask line")
    return s.replace(old, old +
                     "            self.frame_mask = self.frame_mask.cuda()\n"
                     "            if self.coord_mask is not None:\n"
                     "                self.coord_mask = self.coord_mask.cuda()\n"
                     "            if self.frame_lengths is not None:\n"
                     "                self.frame_lengths = self.frame_lengths.cuda()\n")


# ------------------------------------------------------------------------------------ loss.py
NEW_LOSS_CODE = '''def build_element_mask(targets, frame_mask=None, coord_mask=None):
    """
    Boolean mask [batch, time, width] of the target elements that may contribute to the loss.
      frame_mask  [batch, time]         False for padded frames
      coord_mask  [batch, time, width]  False for invalid landmark coordinates. A mask that is one
                                        column narrower than the targets (225 vs 226) leaves the
                                        frame-counter column valid.
    """
    batch, steps, width = targets.shape
    mask = torch.ones((batch, steps, width), dtype=torch.bool, device=targets.device)
    if frame_mask is not None:
        if frame_mask.dim() != 2 or tuple(frame_mask.shape) != (batch, steps):
            raise ValueError("frame_mask must have shape [batch, time] = %s, got %s"
                             % ((batch, steps), tuple(frame_mask.shape)))
        mask = mask & frame_mask.to(device=targets.device, dtype=torch.bool).unsqueeze(-1)
    if coord_mask is not None:
        cm = coord_mask.to(device=targets.device, dtype=torch.bool)
        if cm.dim() != 3 or tuple(cm.shape[:2]) != (batch, steps):
            raise ValueError("coord_mask must have shape [batch, time, width] with batch/time = %s, got %s"
                             % ((batch, steps), tuple(cm.shape)))
        if cm.shape[2] == width - 1:
            cm = torch.cat([cm, torch.ones((batch, steps, 1), dtype=torch.bool, device=targets.device)], dim=2)
        elif cm.shape[2] != width:
            raise ValueError("coord_mask width %d does not match target width %d (or %d without the counter)"
                             % (cm.shape[2], width, width - 1))
        mask = mask & cm
    return mask


class RegLoss(nn.Module):
    """
    Regression loss. With `frame_mask` and/or `coord_mask` only valid elements contribute: padded frames
    and unreliable landmark coordinates are excluded, the mean is taken over the valid elements, and an
    all-invalid batch gives exactly 0 (never a division by zero). Without masks the original behaviour
    (zeros in the target are padding) is kept for regression testing.
    """

    def __init__(self, cfg, target_pad=0.0):
        super(RegLoss, self).__init__()

        self.loss = cfg["training"]["loss"].lower()

        if self.loss == "l1":
            self.criterion = nn.L1Loss(reduction="none")
        elif self.loss == "mse":
            self.criterion = nn.MSELoss(reduction="none")
        else:
            print("Loss not found - revert to default L1 loss")
            self.criterion = nn.L1Loss(reduction="none")

        model_cfg = cfg["model"]

        self.target_pad = target_pad
        self.loss_scale = model_cfg.get("loss_scale", 1.0)

    # pylint: disable=arguments-differ
    def forward(self, preds, targets, frame_mask=None, coord_mask=None):

        if frame_mask is None and coord_mask is None:
            # legacy behaviour: every zero in the target is treated as padding
            loss_mask = (targets != self.target_pad).to(preds.dtype)
            loss = self.criterion(preds * loss_mask, targets * loss_mask).mean()
        else:
            if preds.shape != targets.shape:
                raise ValueError("predictions %s and targets %s differ in shape"
                                 % (tuple(preds.shape), tuple(targets.shape)))
            keep = build_element_mask(targets, frame_mask, coord_mask)
            n_valid = keep.sum().clamp_min(1)
            loss = self.criterion(preds, targets).masked_fill(~keep, 0.0).sum() / n_valid

        if self.loss_scale != 1.0:
            loss = loss * self.loss_scale

        return loss

'''


def patch_loss(s, ctx):
    start = s.index("class RegLoss(")
    end = s.index("class XentLoss(")
    assert start < end, "unexpected class order in loss.py"
    s = s[:start] + NEW_LOSS_CODE + s[end:]
    old = "from torch import nn, Tensor"
    assert old in s, "torch import line not found in loss.py"
    return s.replace(old, "import torch\n" + old, 1)


# ------------------------------------------------------------------------------------ model.py
def patch_model(s, ctx):
    old = "        batch_loss = loss_function(skel_out, batch.trg)\n"
    _once(s, old, "loss call in get_loss_for_batch")
    new = (f"        {MARK} pass the validity masks so padded frames / unreliable landmarks never enter the loss\n"
           "        frame_mask = getattr(batch, \"frame_mask\", None)\n"
           "        coord_mask = getattr(batch, \"coord_mask\", None)\n"
           "        if (frame_mask is not None or coord_mask is not None) and self.future_prediction == 0:\n"
           "            batch_loss = loss_function(skel_out, batch.trg, frame_mask=frame_mask, coord_mask=coord_mask)\n"
           "        else:\n"
           "            batch_loss = loss_function(skel_out, batch.trg)\n")
    return s.replace(old, new)


# ------------------------------------------------------------------------------------ search.py
def patch_search(s, ctx):
    new_block = [
        f"        {MARK} nothing is padded while generating, so every generated frame is visible",
        "        padding_mask = torch.ones(",
        "            (ys.size(0), 1, ys.size(1), ys.size(1)), dtype=torch.bool, device=ys.device)",
    ]
    return _replace_block(s, "# Cut padding mask to required size", "padding_mask = (F.pad(", new_block, "search.py")


PATCHES = [
    ("vocabulary.py", patch_vocabulary),
    ("helpers.py", patch_helpers),
    ("transformer_layers.py", patch_transformer_layers),
    ("batch.py", patch_batch),
    ("loss.py", patch_loss),
    ("model.py", patch_model),
    ("search.py", patch_search),
]


def apply_all(repo_dir, pe_max_len, backup_dir=None, log=print):
    """Back up and patch the repository files in place. Safe to call again (already patched files are skipped)."""
    repo_dir = Path(repo_dir)
    backup_dir = Path(backup_dir) if backup_dir else repo_dir / "_originals"
    backup_dir.mkdir(parents=True, exist_ok=True)
    ctx = {"pe_max_len": int(pe_max_len)}
    messages = []
    for filename, fn in PATCHES:
        path = repo_dir / filename
        src = path.read_text(encoding="utf-8")
        if MARK in src:
            messages.append(f"{filename:24s} already patched")
            continue
        backup = backup_dir / filename
        if not backup.exists():
            shutil.copy2(path, backup)
        try:
            new = fn(src, ctx)
        except AssertionError as exc:
            raise AssertionError(f"Patch for {filename} did not apply: {exc}") from exc
        path.write_text(new.rstrip("\n") + f"\n\n{MARK} applied\n", encoding="utf-8")
        compile(path.read_text(encoding="utf-8"), filename, "exec")     # syntax check right away
        messages.append(f"{filename:24s} patched (original saved in {backup_dir.name}/)")
    for m in messages:
        log(m)
    return messages

In [ ]:
%%writefile /content/isl_code/isign_data.py
"""
PyTorch data pipeline for iSign shards (replaces the torchtext Field/Iterator code).
Written into the Progressive Transformers repository by the notebook; import as `isign_data`.
"""
import re
from collections import Counter
from functools import partial
from types import SimpleNamespace

import numpy as np
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, Sampler

import isign_pose as ip
from constants import DEFAULT_UNK_ID, EOS_TOKEN, PAD_TOKEN
from vocabulary import Vocabulary

_TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def tokenize(text):
    """Lower-case and split into words and punctuation marks."""
    return _TOKEN_RE.findall(text.lower())


class ISignShardDataset(Dataset):
    """
    Sequences stored by isign_pose.convert_manifest. Each item holds
      src            list of tokens
      trg            float32 [T, 226]   (225 normalised coordinates + frame counter)
      landmark_valid bool    [T, 75]    (False where a landmark is missing / unreliable)
      file_paths     the uid
    """

    def __init__(self, shard_dir, normalization=None, max_shards=None, max_frames=None):
        expected = ip.params_hash(normalization) if normalization is not None else None
        self.store = ip.ShardStore(shard_dir, expected_hash=expected, max_shards=max_shards)
        if len(self.store) == 0:
            raise ValueError(f"No converted shards found in {shard_dir}")
        if max_frames is not None and int(self.store.lengths.max()) > max_frames:
            raise ValueError(f"A sequence has {int(self.store.lengths.max())} frames (> max_frames={max_frames})")
        self._tokens = None

    def __len__(self):
        return len(self.store)

    def token_lists(self):
        if self._tokens is None:
            self._tokens = [tokenize(t) for t in self.store.texts]
        return self._tokens

    def __getitem__(self, idx):
        uid, text, coords, valid = self.store.get(int(idx))
        tokens = self.token_lists()[int(idx)] if self._tokens is not None else tokenize(text)
        return {
            "src": tokens,
            "trg": torch.from_numpy(ip.add_counter(coords)),
            "landmark_valid": torch.from_numpy(valid),
            "file_paths": uid,
        }


def build_vocab(token_lists, min_freq=1, max_size=None):
    """Source vocabulary (specials + words), most frequent first."""
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)
    words = [t for t, c in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0])) if c >= min_freq]
    if max_size is not None:
        words = words[:max_size]
    return Vocabulary(tokens=words)


def collate_isign(examples, vocab):
    """Pad examples into the batch object that the repository's Batch class expects."""
    unk_id = DEFAULT_UNK_ID()
    pad_id = vocab.stoi[PAD_TOKEN]
    eos_id = vocab.stoi[EOS_TOKEN]

    src_seqs = [torch.tensor([vocab.stoi.get(t, unk_id) for t in ex["src"]] + [eos_id], dtype=torch.long)
                for ex in examples]
    src_lengths = torch.tensor([len(s) for s in src_seqs], dtype=torch.long)
    src = pad_sequence(src_seqs, batch_first=True, padding_value=pad_id)

    frame_lengths = torch.tensor([ex["trg"].shape[0] for ex in examples], dtype=torch.long)
    trg = pad_sequence([ex["trg"] for ex in examples], batch_first=True, padding_value=0.0)
    # padded frames are 'not valid' for every landmark
    landmark_valid = pad_sequence([ex["landmark_valid"].to(torch.uint8) for ex in examples],
                                  batch_first=True, padding_value=0).bool()

    return SimpleNamespace(
        src=(src, src_lengths),
        trg=trg,
        frame_lengths=frame_lengths,
        landmark_valid=landmark_valid,
        file_paths=[ex["file_paths"] for ex in examples],
    )


class EpochShuffleSampler(Sampler):
    """Shuffles with seed + epoch, so a resumed run sees the same order as an uninterrupted one."""

    def __init__(self, n, seed):
        # no super().__init__(): Sampler's constructor does nothing, and its signature differs between torch versions
        self.n, self.seed, self.epoch = int(n), int(seed), 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __iter__(self):
        return iter(np.random.RandomState(self.seed + self.epoch).permutation(self.n).tolist())

    def __len__(self):
        return self.n


def make_loader(dataset, vocab, batch_size=8, shuffle=False, seed=27, num_workers=0, pin_memory=False):
    sampler = EpochShuffleSampler(len(dataset), seed) if shuffle else None
    # the workers only need the token -> id table; a plain dict pickles under every start method (spawn / forkserver)
    lite_vocab = SimpleNamespace(stoi=dict(vocab.stoi))
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler, shuffle=False,
                      collate_fn=partial(collate_isign, vocab=lite_vocab),
                      num_workers=num_workers, pin_memory=pin_memory,
                      persistent_workers=num_workers > 0)       # on Windows every new worker process costs seconds

In [ ]:
%%writefile /content/isl_code/isign_train.py
"""
Device handling, batching helpers, training / evaluation loops and checkpointing for the iSign
text-to-pose experiments. Written into the Progressive Transformers repository by the notebook;
import as `isign_train`. Uses the repository's Model / Batch / RegLoss (after the notebook's patches).
"""
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch

from batch import Batch
from loss import build_element_mask

BATCH_TENSOR_ATTRS = ("src", "src_lengths", "src_mask", "trg_input", "trg", "trg_mask",
                      "frame_mask", "coord_mask", "frame_lengths")


# ----------------------------------------------------------------------------- reproducibility / device
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def select_device(require_cuda=False):
    """
    Return (device, info). With require_cuda=True a missing GPU raises a clear error; otherwise the
    CPU is used (fine for preprocessing and small smoke tests, not for training).
    """
    info = {"torch": torch.__version__, "cuda_build": torch.version.cuda,
            "cuda_available": bool(torch.cuda.is_available())}
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
        props = torch.cuda.get_device_properties(0)
        info.update({"gpu_name": props.name, "gpu_memory_gb": round(props.total_memory / 1e9, 1),
                     "gpu_capability": f"{props.major}.{props.minor}"})
    else:
        if require_cuda:
            raise RuntimeError(
                "No CUDA device is available. In Colab choose Runtime > Change runtime type > T4 GPU, "
                "then restart and run again. (CPU is only acceptable for preprocessing and smoke tests.)")
        device = torch.device("cpu")
    info["device"] = str(device)
    return device, info


def prepare_model(model, device):
    """Put the model on `device` and make the repository's own use_cuda flag agree with it."""
    model.to(device)
    model.use_cuda = device.type == "cuda"
    return model


def parameter_devices(model):
    return sorted({str(p.device) for p in model.parameters()})


# ----------------------------------------------------------------------------- batches
def move_batch(batch, device):
    for name in BATCH_TENSOR_ATTRS:
        value = getattr(batch, name, None)
        if torch.is_tensor(value) and value.device != device:
            setattr(batch, name, value.to(device))
    return batch


def make_batch(raw, pad_index, model, device):
    batch = Batch(torch_batch=raw, pad_index=pad_index, model=model)
    return move_batch(batch, device)


def batch_devices(batch):
    return {name: str(getattr(batch, name).device) for name in BATCH_TENSOR_ATTRS
            if torch.is_tensor(getattr(batch, name, None))}


def assert_batch_device(batch, device):
    placed = batch_devices(batch)
    wrong = {k: v for k, v in placed.items()
             if torch.device(v).type != device.type or (device.index is not None and torch.device(v).index != device.index)}
    assert not wrong, f"batch tensors not on {device}: {wrong}"
    return placed


def count_valid_elements(batch):
    """Number of target elements that may contribute to the loss (padded frames and invalid landmarks excluded)."""
    mask = build_element_mask(batch.trg, frame_mask=batch.frame_mask, coord_mask=batch.coord_mask)
    return int(mask.sum().item())


# ----------------------------------------------------------------------------- one optimisation step
def train_step(model, batch, criterion, optimizer, clip_norm=None):
    """Forward, masked loss, backward, clip, step. Skips (and reports) non-finite losses / gradients."""
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss, _ = model.get_loss_for_batch(batch, criterion)
    if not bool(torch.isfinite(loss).item()):
        optimizer.zero_grad(set_to_none=True)
        return {"loss": float("nan"), "grad_norm": float("nan"), "skipped": "non-finite loss"}
    loss.backward()
    if clip_norm:
        grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), float(clip_norm)))
    else:
        norms = [p.grad.detach().norm() for p in model.parameters() if p.grad is not None]
        grad_norm = float(torch.norm(torch.stack(norms))) if norms else 0.0
    if not math.isfinite(grad_norm):
        optimizer.zero_grad(set_to_none=True)
        return {"loss": float(loss.item()), "grad_norm": grad_norm, "skipped": "non-finite gradient"}
    optimizer.step()
    return {"loss": float(loss.item()), "grad_norm": grad_norm, "skipped": None}


def run_train_epoch(model, loader, criterion, optimizer, pad_index, device, clip_norm=None):
    stats = {"loss_sum": 0.0, "n_valid": 0, "steps": 0, "skipped_empty": 0, "skipped_nonfinite": 0,
             "max_grad_norm": 0.0}
    start = time.time()
    for raw in loader:
        batch = make_batch(raw, pad_index, model, device)
        n_valid = count_valid_elements(batch)
        if n_valid == 0:                      # nothing to learn from: skip explicitly, never divide by zero
            stats["skipped_empty"] += 1
            continue
        result = train_step(model, batch, criterion, optimizer, clip_norm)
        if result["skipped"]:
            stats["skipped_nonfinite"] += 1
            continue
        stats["loss_sum"] += result["loss"] * n_valid
        stats["n_valid"] += n_valid
        stats["steps"] += 1
        stats["max_grad_norm"] = max(stats["max_grad_norm"], result["grad_norm"])
    stats["loss"] = stats["loss_sum"] / stats["n_valid"] if stats["n_valid"] else float("nan")
    stats["seconds"] = time.time() - start
    return stats


@torch.no_grad()
def evaluate(model, loader, criterion, pad_index, device, free_running=False, max_batches=None):
    """
    Mean loss over valid elements (frames and landmarks), gradients disabled.
    free_running=True decodes greedily (ground-truth frame counters drive the timing, as in the
    original repository's validation) and scores the generated frames instead of the teacher-forced ones.
    """
    model.eval()
    total, n_valid_total, n_batches, n_seq, empty = 0.0, 0, 0, 0, 0
    for i, raw in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        batch = make_batch(raw, pad_index, model, device)
        n_seq += batch.nseqs
        n_valid = count_valid_elements(batch)
        if n_valid == 0:
            empty += 1
            continue
        if free_running:
            generated, _ = model.run_batch(batch, max_output_length=None)
            loss = criterion(generated[:, 1:, :], batch.trg, frame_mask=batch.frame_mask, coord_mask=batch.coord_mask)
        else:
            loss, _ = model.get_loss_for_batch(batch, criterion)
        total += float(loss.item()) * n_valid
        n_valid_total += n_valid
        n_batches += 1
    return {"loss": total / n_valid_total if n_valid_total else float("nan"), "n_valid": n_valid_total,
            "batches": n_batches, "sequences": n_seq, "skipped_empty_batches": empty, "free_running": bool(free_running)}


# ----------------------------------------------------------------------------- checkpoints
def save_checkpoint(path, model, optimizer=None, scheduler=None, state=None, normalization=None, cfg=None):
    payload = {"model_state": model.state_dict(), "state": state or {}, "normalization": normalization,
               "config": cfg, "torch_version": torch.__version__}
    if optimizer is not None:
        payload["optimizer_state"] = optimizer.state_dict()
    if scheduler is not None:
        payload["scheduler_state"] = scheduler.state_dict()
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)          # atomic: an interrupted save never leaves a half-written checkpoint


def load_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)   # our own files only
    except TypeError:   # very old torch without weights_only
        return torch.load(path, map_location=device)


# ----------------------------------------------------------------------------- full training loop
def train(model, train_loader, val_loader, criterion, cfg, run_dir, device, pad_index,
          normalization=None, max_epochs=None, resume=True, log=print, time_budget_s=None):
    """
    Epoch-based training with validation, best-checkpoint selection on validation loss, LR plateau
    scheduling, early stopping and resume from run_dir/last.ckpt (granularity: completed epochs).
    time_budget_s: stop cleanly after the epoch during which this many seconds have passed (the run stays
    resumable and model selection is NOT marked finished). The test set is never touched here.
    """
    session_start = time.time()
    tr = cfg["training"]
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    max_epochs = int(max_epochs or tr["epochs"])
    optimizer = torch.optim.Adam(model.parameters(), lr=float(tr["learning_rate"]),
                                 weight_decay=float(tr.get("weight_decay", 0.0)))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=float(tr["decrease_factor"]), patience=int(tr["patience"]))
    lr_stop = float(tr.get("learning_rate_min", 0.0))
    early_patience = int(tr.get("early_stop_patience", 10))
    val_every = int(tr.get("val_every_epochs", 1))
    clip_norm = tr.get("clip_grad_norm")

    state = {"epoch": 0, "global_step": 0, "best_val": float("inf"), "best_epoch": 0,
             "bad_validations": 0, "history": [], "stop_reason": None}
    last_path, best_path = run_dir / "last.ckpt", run_dir / "best.ckpt"
    if resume and last_path.exists():
        ckpt = load_checkpoint(last_path, device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        state = ckpt["state"]
        log(f"Resumed from {last_path.name}: {state['epoch']} epochs done, best val {state['best_val']:.5f} "
            f"(epoch {state['best_epoch']}), stop reason so far: {state['stop_reason']}")
        if state.get("stop_reason") in ("early_stopping", "lr_below_min"):
            log("This run had already stopped; nothing to do.")
            return state
        state["stop_reason"] = None

    sampler = getattr(train_loader, "sampler", None)
    if state["epoch"] < max_epochs:             # training continues, so an earlier "selection finished" marker is stale
        (run_dir / "selection_done.json").unlink(missing_ok=True)
    for epoch in range(state["epoch"] + 1, max_epochs + 1):
        if hasattr(sampler, "set_epoch"):
            sampler.set_epoch(epoch)
        stats = run_train_epoch(model, train_loader, criterion, optimizer, pad_index, device, clip_norm)
        entry = {"epoch": epoch, "train_loss": stats["loss"], "lr": optimizer.param_groups[0]["lr"],
                 "steps": stats["steps"], "skipped_empty": stats["skipped_empty"],
                 "skipped_nonfinite": stats["skipped_nonfinite"], "max_grad_norm": stats["max_grad_norm"],
                 "seconds": round(stats["seconds"], 1)}
        state["global_step"] += stats["steps"]
        stop = None
        if epoch % val_every == 0:
            val = evaluate(model, val_loader, criterion, pad_index, device)
            entry["val_loss"] = val["loss"]
            improved = False
            if math.isfinite(val["loss"]):
                scheduler.step(val["loss"])
                improved = val["loss"] < state["best_val"] - 1e-8
            if improved:
                state.update(best_val=val["loss"], best_epoch=epoch, bad_validations=0)
                save_checkpoint(best_path, model, state=dict(state), normalization=normalization, cfg=cfg)
            else:
                state["bad_validations"] += 1
            if state["bad_validations"] >= early_patience:
                stop = "early_stopping"
            if optimizer.param_groups[0]["lr"] < lr_stop:
                stop = "lr_below_min"
        state["epoch"] = epoch
        state["history"].append(entry)
        state["stop_reason"] = stop or ("max_epochs" if epoch >= max_epochs else None)
        save_checkpoint(last_path, model, optimizer, scheduler, state, normalization, cfg)
        log(f"epoch {epoch:3d} | train {entry['train_loss']:.5f} | val {entry.get('val_loss', float('nan')):.5f} | "
            f"lr {entry['lr']:.2e} | steps {entry['steps']} | skipped empty/non-finite "
            f"{entry['skipped_empty']}/{entry['skipped_nonfinite']} | best epoch {state['best_epoch']}")
        if stop:
            break
        if time_budget_s is not None and time.time() - session_start >= time_budget_s and epoch < max_epochs:
            log(f"time budget of {time_budget_s / 3600:.1f} h used up after epoch {epoch}; stopping here. "
                "Run the same command again to resume.")
            state["paused"] = True
            break

    if state["stop_reason"] is None and state["epoch"] >= max_epochs:
        state["stop_reason"] = "max_epochs"
    (run_dir / "history.json").write_text(json.dumps(state["history"], indent=1), encoding="utf-8")
    if state["stop_reason"]:
        (run_dir / "selection_done.json").write_text(json.dumps({
            "best_epoch": state["best_epoch"], "best_val": state["best_val"], "stop_reason": state["stop_reason"],
            "epochs_run": state["epoch"], "normalization": normalization}), encoding="utf-8")
    return state


def final_test_evaluation(model, test_loader, criterion, run_dir, device, pad_index, free_running=True, force=False,
                          free_running_max_batches=None):
    """
    Score the BEST validation checkpoint on the test split. Allowed once, and only after training has finished.
    The teacher-forced score covers the whole split; the (slow) free-running score can be limited to the first
    free_running_max_batches batches.
    """
    run_dir = Path(run_dir)
    if not (run_dir / "selection_done.json").exists():
        raise RuntimeError("Model selection has not finished (no selection_done.json); the test set must not be used yet.")
    out_path = run_dir / "test_results.json"
    if out_path.exists() and not force:
        raise RuntimeError(f"{out_path.name} already exists. Re-evaluating on the test set after looking at it "
                           "is model selection on the test set; pass force=True only if you really intend that.")
    ckpt = load_checkpoint(run_dir / "best.ckpt", device)
    model.load_state_dict(ckpt["model_state"])
    results = {"best_epoch": ckpt["state"].get("best_epoch"),
               "teacher_forced": evaluate(model, test_loader, criterion, pad_index, device, free_running=False)}
    if free_running:
        results["free_running"] = evaluate(model, test_loader, criterion, pad_index, device, free_running=True,
                                           max_batches=free_running_max_batches)
    out_path.write_text(json.dumps(results, indent=1), encoding="utf-8")
    return results


# ----------------------------------------------------------------------------- overfit helper
def overfit_steps(model, loader, criterion, cfg, pad_index, device, steps, log_every=10, log=print):
    """A fixed number of optimiser steps on one small loader (no validation / test data involved)."""
    tr = cfg["training"]
    optimizer = torch.optim.Adam(model.parameters(), lr=float(tr["learning_rate"]),
                                 weight_decay=float(tr.get("weight_decay", 0.0)))
    sampler = getattr(loader, "sampler", None)
    epoch, history = 1, []
    if hasattr(sampler, "set_epoch"):
        sampler.set_epoch(epoch)
    iterator = iter(loader)
    for step in range(1, steps + 1):
        try:
            raw = next(iterator)
        except StopIteration:            # next pass over the small subset, reshuffled
            epoch += 1
            if hasattr(sampler, "set_epoch"):
                sampler.set_epoch(epoch)
            iterator = iter(loader)
            raw = next(iterator)
        batch = make_batch(raw, pad_index, model, device)
        if count_valid_elements(batch) == 0:
            continue
        result = train_step(model, batch, criterion, optimizer, tr.get("clip_grad_norm"))
        history.append(result)
        if log_every and step % log_every == 0:
            log(f"step {step:4d} | loss {result['loss']:.5f} | grad norm {result['grad_norm']:.3f}")
    return history

## 4. Environment, repository clone and patches

In [ ]:
# CELL 2 — Install the pose reader, load the project modules, report the runtime.
# If Colab asks to restart the session after this cell, restart and run again from CELL 1.
%pip install -q pose-format

import importlib.metadata as md
import torch

for _pkg in ("pose-format", "numpy", "pandas", "torch"):
    try:
        print(f"{_pkg:12s}", md.version(_pkg))
    except Exception:
        print(f"{_pkg:12s} version unknown")
print("CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU in this runtime")

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for _m in ("isign_pose", "isign_report", "repo_patches"):
    sys.modules.pop(_m, None)
importlib.invalidate_caches()
import isign_pose as ip
import isign_report as rep
import repo_patches as rp
from pose_format import Pose

CHECKS = rep.Checks()

In [ ]:
# CELL 3 — Clone a pristine copy of the repository and apply the asserted patches (originals are backed up).
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
head = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
print("Repository commit:", head)
if head != rp.TESTED_COMMIT:
    print(f"WARNING: patches were written for {rp.TESTED_COMMIT}; if the next step raises an AssertionError the repository changed.")

rp.apply_all(REPO_DIR, PE_MAX_LEN)
print()
print(subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"], capture_output=True, text=True).stdout)
print("Originals of every modified file:", sorted(p.name for p in (REPO_DIR / "_originals").glob("*.py")))

## 5. Archive access and UID matching

In [ ]:
# CELL 4 — Open the split pose archive through the virtual concatenation (nothing is combined or extracted).
archive = virtual_file = None
pose_names, pose_by_stem, KEEP_IDX = [], {}, ip.EXPECTED_KEEP_IDX

with CHECKS("1. Dataset archive access (no combined archive)") as chk:
    parts = [POSE_DIR / n for n in POSE_PART_NAMES]
    absent = [str(p) for p in parts if not p.exists()]
    assert not absent, f"pose parts missing on Drive: {absent}"
    virtual_file = ip.VirtualConcatFile(parts)
    assert virtual_file.total_size == sum(p.stat().st_size for p in parts)
    archive = zipfile.ZipFile(virtual_file, "r")
    pose_names = [n for n in archive.namelist() if n.lower().endswith(".pose")]
    assert pose_names, "no .pose entries in the archive"
    pose_by_stem = {PurePosixPath(n).stem: n for n in pose_names}
    first_pose = Pose.read(archive.read(pose_names[0]))
    KEEP_IDX, layout_note = ip.resolve_landmark_layout(first_pose)
    print("Landmark layout:", layout_note)
    # no multi-GB file may exist on the local disk (outside the mounted Drive)
    large = []
    for root, dirs, files in os.walk("/content"):
        dirs[:] = [d for d in dirs if d != "drive"]
        for f in files:
            fp = os.path.join(root, f)
            try:
                if os.path.getsize(fp) > 1e9:
                    large.append(fp)
            except OSError:
                pass
    assert not large, f"large local files found (a combined archive?): {large}"
    chk.note(f"{len(pose_names):,} pose entries from {len(parts)} parts ({virtual_file.total_size / 1e9:.1f} GB), read in place")
print(f"Archive entries: {len(pose_names):,} | local disk free: {shutil.disk_usage('/').free / 1e9:.0f} GB")

In [ ]:
# CELL 5 — Match CSV uids to pose files and log everything that does not match.
df = pd.read_csv(CSV_PATH)
assert {"uid", "text"} <= set(df.columns), f"Unexpected CSV columns: {df.columns.tolist()}"
df["uid"] = df["uid"].astype(str)
n_rows = len(df)
duplicate_uids = df.loc[df["uid"].duplicated(), "uid"].tolist()
df = df.drop_duplicates("uid", keep="first").reset_index(drop=True)
df["text_norm"] = df["text"].astype(str).map(ip.normalize_text)
df["video_id"] = df["uid"].map(ip.video_id_of)

stem_set = set(pose_by_stem)
has_pose = df["uid"].isin(stem_set)
empty_text = df.loc[df["text_norm"] == "", "uid"].tolist()
unmatched_csv = df.loc[~has_pose, "uid"].tolist()
pose_without_csv = sorted(stem_set - set(df["uid"]))
matched = df[has_pose & (df["text_norm"] != "")].reset_index(drop=True)

LOG_DIR = DRIVE_OUT / "logs"
LOG_DIR.mkdir(exist_ok=True)
for fname, items in (("unmatched_csv_uids.txt", unmatched_csv), ("pose_files_without_csv_row.txt", pose_without_csv),
                     ("empty_text_uids.txt", empty_text), ("duplicate_uids.txt", duplicate_uids)):
    (LOG_DIR / fname).write_text("\n".join(items) + ("\n" if items else ""), encoding="utf-8")

print(f"CSV rows: {n_rows:,} | duplicates dropped: {len(duplicate_uids)} | empty text: {len(empty_text)}")
print(f"matched to a pose file: {len(matched):,} ({len(matched) / max(len(df), 1):.1%}) | "
      f"CSV uids without a pose file: {len(unmatched_csv):,} | pose files without a CSV row: {len(pose_without_csv):,}")
print("first unmatched CSV uids:", unmatched_csv[:5])
print("first pose files without CSV row:", pose_without_csv[:5])
print("logs written to", LOG_DIR)

with CHECKS("2. UID-to-pose matching") as chk:
    rate = len(matched) / max(len(df), 1)
    assert rate >= MIN_MATCH_RATE, f"only {rate:.1%} of CSV rows have a pose file (< {MIN_MATCH_RATE:.0%}); inspect {LOG_DIR / 'unmatched_csv_uids.txt'}"
    assert matched["uid"].is_unique
    rng = np.random.RandomState(SEED)
    for i in rng.choice(len(matched), min(10, len(matched)), replace=False):
        p = Pose.read(archive.read(pose_by_stem[matched.loc[i, "uid"]]))
        assert p.body.data.shape[0] > 0, f"{matched.loc[i, 'uid']}: pose has no frames"
    chk.note(f"{len(matched):,}/{len(df):,} matched ({rate:.1%}); {len(unmatched_csv)} CSV uids and "
             f"{len(pose_without_csv)} pose files unmatched (logged); 10 random matched poses read OK")

## 6. Train / validation / test split by video ID
UIDs look like `<video_id>-<sequence_number>`; the video ID is everything before the last `-`. Whole videos are assigned to one split, using a fixed seed. The split is decided on the **matched** sequences before any quality filtering, so later exclusions never move a video between splits.

In [ ]:
# CELL 6 — Build the split manifests and report their statistics.
uids_by_split, videos_by_split = ip.split_by_video(matched["uid"].tolist(), SEED, SPLIT_FRACTIONS, SPLITS)
text_of = dict(zip(matched["uid"], matched["text_norm"]))
manifests = {s: pd.DataFrame({"uid": u, "video_id": [ip.video_id_of(x) for x in u], "text": [text_of[x] for x in u]})
             for s, u in uids_by_split.items()}
rows_of = {s: list(zip(m["uid"], m["text"])) for s, m in manifests.items()}

# Reading the archive in the order the entries are stored makes the reads (almost) sequential, which Drive serves far
# faster than random access. The split membership above is unaffected; only the order of conversion changes.
offset_of = {u: archive.getinfo(pose_by_stem[u]).header_offset for u in matched["uid"]}
convert_rows_of = {s: sorted(rows, key=lambda r: offset_of[r[0]]) for s, rows in rows_of.items()}

SPLIT_DIR = DRIVE_OUT / "splits"
SPLIT_DIR.mkdir(exist_ok=True)
for s, m in manifests.items():
    m.to_csv(SPLIT_DIR / f"manifest_{s}.csv", index=False)

split_table = pd.DataFrame(ip.split_report(uids_by_split, videos_by_split, SPLIT_FRACTIONS, len(matched)))
split_table["share_of_sequences"] = split_table["share_of_sequences"].map("{:.1%}".format)
split_table["target_share"] = split_table["target_share"].map("{:.0%}".format)
display(split_table)
print("manifests written to", SPLIT_DIR)

with CHECKS("8. Train/val/test separation by video ID") as chk:
    assert not ip.split_overlaps(videos_by_split), "a video occurs in more than one split"
    vid_sets = {s: set(m["video_id"]) for s, m in manifests.items()}
    uid_sets = {s: set(m["uid"]) for s, m in manifests.items()}
    for a, b in itertools.combinations(SPLITS, 2):
        assert not (vid_sets[a] & vid_sets[b]), f"{a}/{b} share videos"
        assert not (uid_sets[a] & uid_sets[b]), f"{a}/{b} share uids"
    assert sum(len(u) for u in uid_sets.values()) == len(matched), "splits do not cover every matched sequence exactly once"
    again, _ = ip.split_by_video(matched["uid"].tolist(), SEED, SPLIT_FRACTIONS, SPLITS)
    assert again == uids_by_split, "split is not reproducible for a fixed seed"
    for row in ip.split_report(uids_by_split, videos_by_split, SPLIT_FRACTIONS, len(matched)):
        assert row["sequences"] > 0, f"{row['split']} is empty"
        assert abs(row["share_of_sequences"] - row["target_share"]) < 0.05, f"{row['split']} share {row['share_of_sequences']:.1%} is far from the target"
    chk.note("; ".join(f"{s}: {len(uid_sets[s]):,} seq / {len(vid_sets[s]):,} videos" for s in SPLITS))

## 7. Coordinate and confidence diagnostics
A random sample of **training-split** sequences is read straight from the archive (no quality filter) and examined before the normalisation is frozen. If `Z_MODE = "scale"`, the z scales are estimated from this training sample only and then reused for every split and for inference.

In [ ]:
# CELL 7 — Read a training-split sample and extract the raw landmark values, confidences and missing-data categories.
rng = np.random.RandomState(SEED)
train_manifest = manifests["train"]
pick = rng.choice(len(train_manifest), min(DIAG_N_SEQUENCES, len(train_manifest)), replace=False)
pick = sorted(pick, key=lambda i: offset_of[train_manifest.loc[int(i), "uid"]])    # read in archive order (still a random sample)
diag_raws, diag_uids, diag_unreadable = [], [], 0
t0 = time.time()
for n, i in enumerate(pick, 1):
    uid = train_manifest.loc[int(i), "uid"]
    try:
        raw = ip.extract_raw(Pose.read(archive.read(pose_by_stem[uid])), KEEP_IDX)
    except Exception:
        raw = None
    if raw is None:
        diag_unreadable += 1
        continue
    diag_raws.append(raw)
    diag_uids.append(uid)
    if n % 50 == 0:
        print(f"  read {n}/{len(pick)} sequences ({time.time() - t0:.0f}s)")
print(f"{len(diag_raws)} sequences usable for diagnostics, {diag_unreadable} unreadable")
assert len(diag_raws) >= 30, "too few readable sequences for meaningful diagnostics"

In [ ]:
# CELL 8 — Distributions, validity categories, coordinate-system compatibility, z scale, discontinuities.
params_keep = ip.make_normalization("keep", None, CONF_THRESHOLD, ZERO_FILL_IS_MISSING, STORAGE_DTYPE)
if Z_MODE == "scale":
    z_scales = ip.estimate_z_scales(diag_raws, params_keep)      # TRAINING sample only
    NORM = ip.make_normalization("scale", z_scales, CONF_THRESHOLD, ZERO_FILL_IS_MISSING, STORAGE_DTYPE)
    print("Estimated z scales (train sample, 99th percentile of |z|):", {k: round(v, 4) for k, v in z_scales.items()})
else:
    NORM = params_keep
NORM_HASH = ip.params_hash(NORM)

report, flags = ip.diagnose_sample(diag_raws, NORM)
DIAG_REPORT = report

print("=== Landmark validity (all categories overlap; 'valid' is what remains) ===")
print(json.dumps(report["landmark_breakdown"], indent=1))
print("\n=== Confidence ===")
print(json.dumps(report["confidence"], indent=1))

rows = []
for comp, r in report["components"].items():
    for axis in ("x_raw", "y_raw", "z_raw"):
        rows.append({"component": comp, "axis": axis[0], "n": r["n_valid_landmarks"], **r[axis]})
print("\n=== Raw value percentiles per component (valid landmarks only; p0 = min, p100 = max) ===")
display(pd.DataFrame(rows).round(3))
print("\nImage sizes in the sample (width, height, sequences):", [(s["width"], s["height"], s["sequences"]) for s in report["image_sizes"]])
print("x/y are normalised per sequence with its own image size:", NORM["xy"]["formula"])
print("Percent of valid x/y outside their own image:",
      {c: (None if r["pct_xy_outside_nominal_bounds"] is None else round(r["pct_xy_outside_nominal_bounds"], 2))
       for c, r in report["components"].items()})
print("\n=== Which landmark groups carry the target variance? (share of the squared error left if each landmark were predicted by its mean) ===")
display(pd.DataFrame(report["landmark_groups"]).round(1))
print("z spread / x spread (IQR ratio, effective z):",
      {c: (None if r["z_iqr_over_x_iqr"] is None else round(r["z_iqr_over_x_iqr"], 3)) for c, r in report["components"].items()})
print("Hand-wrist z, median |z| (should be ~0 if hand z is wrist-relative):", report["hand_wrist_z"])
print("\n=== Do hands and body share a coordinate system? Median wrist distance in normalised units ===")
print({k: (None if v["median_norm"] is None else round(v["median_norm"], 3)) for k, v in report["wrist_pairing"].items()},
      "(LL/RR = own wrist, LR/RL = opposite wrist; LL, RR should be the smallest)")
print("\n=== Temporal behaviour ===")
print(json.dumps(report["temporal"], indent=1))

print("\n=== Flags ===")
for sev, msg in flags:
    print(f"[{sev}] {msg}")
if not flags:
    print("none")

Z_RATIOS = {c: r["z_iqr_over_x_iqr"] for c, r in report["components"].items()}
Z_DOMINATES = any(v is not None and v > 10 for v in Z_RATIOS.values())
print(f"\nz mode '{NORM['z']['mode']}': keeping z in source units is {'NOT appropriate: z would dominate the loss' if Z_DOMINATES else 'appropriate: z does not dominate'} "
      f"(z spread / x spread: {({c: None if v is None else round(v, 3) for c, v in Z_RATIOS.items()})}; a small ratio only means z has little weight)")
if Z_DOMINATES and NORM["z"]["mode"] == "keep":
    print("-> set Z_MODE = 'scale' in CELL 1 and rerun from CELL 8; scales are then estimated from the training sample only.")

(DRIVE_OUT / "diagnostics").mkdir(exist_ok=True)
(DRIVE_OUT / "diagnostics" / "diagnostics_report.json").write_text(json.dumps(report, indent=1), encoding="utf-8")

In [ ]:
# CELL 9 — Plots: distributions, skeleton reconstruction, body-wrist vs hand-wrist tracks.
from IPython.display import Image

diag_dir = DRIVE_OUT / "diagnostics"
ip.plot_distributions(diag_raws, NORM, diag_dir / "distributions.png")
ip.plot_skeletons(diag_raws, NORM, diag_dir / "skeletons.png")
ip.plot_wrist_tracks(diag_raws, NORM, diag_dir / "wrist_tracks.png")
for _name in ("distributions", "skeletons", "wrist_tracks"):
    display(Image(filename=str(diag_dir / f"{_name}.png")))
print("Look at the skeleton plots: shoulders, elbows, wrists and both hands should form a coherent upper body, and the "
      "hand wrists should follow the body wrists in the track plot. If they do not, do not proceed.")

In [ ]:
# CELL 10 — Freeze and save the normalisation configuration.
NORM_JSON = json.dumps(NORM, indent=1, sort_keys=True)
(WORK_DIR / "normalization.json").write_text(NORM_JSON, encoding="utf-8")
(DRIVE_OUT / f"normalization_{NORM_HASH}.json").write_text(NORM_JSON, encoding="utf-8")
PROC_ROOT = PROC_PARENT / f"norm_{NORM_HASH}_archive_order"   # a different normalisation or row order can never mix into these shards
PROC_ROOT.mkdir(exist_ok=True)
print("Normalisation hash:", NORM_HASH, "| converted data goes to", PROC_ROOT)
print(NORM_JSON)

## 8. Convert pilot and splits
Sequences are converted to shards (`.text`, `.files`, binary `.skels` coordinates, `.valid` landmark flags, `.idx.npy`, `.json`). The binary `.skels` replaces the text format used for the 5-example pilot: as text, the full dataset would need tens of gigabytes. Counter values are not stored; they are regenerated as `frame_index / n_frames`. Conversion is resumable; shards that exist are skipped. Every excluded sequence is logged with its reason.

**Order and disk.** Within each split, sequences are converted in the order they are stored in the archive, so Drive is read (almost) sequentially instead of at random; the review-scale subset is therefore the first videos of each split in archive order, not a random sample (the coordinate diagnostics above *are* a random sample). The conversion checks the free local disk before every shard and stops cleanly below `MIN_FREE_GB`. Google Drive caches what it reads on the VM's local disk, so run the disk report first if Colab warns that the disk is almost full.

In [ ]:
# CELL 10b — Where is the local disk going? (Drive keeps a cache of files it has read, outside /content.)
def disk_report():
    def sh(cmd):
        return subprocess.run(cmd + " 2>/dev/null", shell=True, capture_output=True, text=True).stdout.strip()
    print(sh("df -h / | tail -n 2"))
    print(f"\nfree on /: {shutil.disk_usage('/').free / 1e9:.1f} GB (conversion stops below {MIN_FREE_GB} GB)\n")
    for path in ("/content", "/root/.cache", "/root/.config/Google/DriveFS", "/tmp"):
        print(sh(f"du -xsh {path}") or f"{path}: not present")
    print("\nlargest folders inside the Drive cache:")
    print(sh("du -xh --max-depth=3 /root/.config/Google/DriveFS | sort -h | tail -n 8") or "none found")


def clear_drivefs_cache(confirm=False):
    """
    UNVERIFIED helper: use only if disk_report() shows that DriveFS's content_cache is what fills the disk.
    Unmounts Drive (flushing pending writes), deletes the cache folders, remounts Drive. Afterwards re-run CELL 4,
    because the archive handles opened before the unmount are no longer valid.
    """
    dirs = sorted(Path("/root/.config/Google/DriveFS").glob("*/content_cache"))
    print("cache folders:", [str(d) for d in dirs] or "none found")
    if not confirm:
        print("Dry run only. Call clear_drivefs_cache(confirm=True) to unmount Drive, delete these folders and remount.")
        return
    drive.flush_and_unmount()
    for d in dirs:
        shutil.rmtree(d, ignore_errors=True)
    drive.mount("/content/drive", force_remount=True)
    print("Drive remounted. Re-run CELL 4 before continuing.")


disk_report()

In [ ]:
# CELL 11 — Convert the isolated pilot and the split shards.
def read_pose_by_uid(uid):
    name = pose_by_stem.get(uid)
    return None if name is None else Pose.read(archive.read(name))


PILOT_DIR = WORK_DIR / "pilot"                   # local, isolated from the split data; never used for evaluation
if PILOT_DIR.exists():
    shutil.rmtree(PILOT_DIR)
pilot_rows = list(zip(matched["uid"].head(PILOT_N * 3), matched["text_norm"].head(PILOT_N * 3)))
print("Pilot:", ip.convert_manifest(pilot_rows, PILOT_DIR, read_pose_by_uid, NORM, QUALITY, shard_size=len(pilot_rows)))

CONVERT_INFO = {}
for _split in SPLITS:
    limit = None if FULL_CONVERT else CONVERT_MAX_SHARDS[_split]
    print(f"\n{_split}: {len(rows_of[_split]):,} planned sequences, converting {'all' if limit is None else limit} shard(s) of {SHARD_SIZE}")
    CONVERT_INFO[_split] = ip.convert_manifest(convert_rows_of[_split], PROC_ROOT / _split, read_pose_by_uid, NORM, QUALITY,
                                               SHARD_SIZE, max_shards=limit, min_free_gb=MIN_FREE_GB)
    if CONVERT_INFO[_split]["stopped"]:
        disk_report()
        raise RuntimeError(f"Conversion of '{_split}' stopped: {CONVERT_INFO[_split]['stopped']}. Finished shards are kept. "
                           "Free space (see the report above and clear_drivefs_cache), then re-run this cell to resume.")

STORES = {s: ip.ShardStore(PROC_ROOT / s, expected_hash=NORM_HASH) for s in SPLITS}
conv_rows, exclusion_rows = [], []
for s, st in STORES.items():
    exc = st.exclusions()
    exclusion_rows += [{"split": s, **e} for e in exc]
    conv_rows.append({"split": s, "planned": len(rows_of[s]), "shards": len(st.metas), "kept": len(st), "excluded": len(exc),
                      "videos_kept": len({ip.video_id_of(u) for u in st.uids}),
                      "frames": int(st.lengths.sum()), "mean_valid_fraction": float(st.valid_fraction.mean()) if len(st) else float("nan")})
display(pd.DataFrame(conv_rows))
if exclusion_rows:
    exc_df = pd.DataFrame(exclusion_rows)
    exc_df["reason_group"] = exc_df["reason"].str.replace(r"\s*\(.*", "", regex=True)
    display(exc_df.groupby(["split", "reason_group"]).size().rename("count").reset_index())
    exc_df.drop(columns="reason_group").to_csv(LOG_DIR / "excluded_sequences.csv", index=False)
    print("every excluded sequence is listed in", LOG_DIR / "excluded_sequences.csv")
else:
    print("no sequences were excluded")

## 9. Data checks

In [ ]:
# CELL 12 — Alignment of text, files and poses; coordinate / confidence validity of the stored data.
def stored_equals_fresh(uid, stored_coords, stored_valid):
    fresh = ip.raw_to_record(ip.extract_raw(read_pose_by_uid(uid), KEEP_IDX), NORM)
    return (fresh["coords"].shape == stored_coords.shape and np.array_equal(fresh["valid"], stored_valid)
            and np.allclose(fresh["coords"], stored_coords, rtol=2e-3, atol=2e-3))


with CHECKS("3. Text/pose/file count alignment") as chk:
    rng = np.random.RandomState(SEED)
    for s, st in STORES.items():
        n = len(st)
        assert n > 0, f"{s}: nothing converted"
        assert len(st.uids) == len(st.texts) == len(st.lengths) == len(st.valid_fraction) == n
        assert len(set(st.uids)) == n, f"{s}: duplicate uids"
        manifest_text = dict(rows_of[s])
        assert set(st.uids) <= set(manifest_text), f"{s}: converted uids that are not in the split manifest"
        bad_text = [u for u, t in zip(st.uids, st.texts) if manifest_text[u] != t]
        assert not bad_text, f"{s}: text differs from the manifest for {bad_text[:3]}"
        assert int(st.lengths.sum()) == sum(m["n_frames"] for m in st.metas), f"{s}: frame counts differ from shard metadata"
        for i in rng.choice(n, min(20, n), replace=False):        # re-read from the archive: uid really maps to this pose
            uid, _, coords, valid = st.get(int(i))
            assert stored_equals_fresh(uid, coords, valid), f"{s}: stored data for {uid} differs from the archive"
    videos_kept = {s: {ip.video_id_of(u) for u in st.uids} for s, st in STORES.items()}
    for a, b in itertools.combinations(SPLITS, 2):
        assert not (videos_kept[a] & videos_kept[b]), f"converted data: {a}/{b} share videos"
    chk.note("; ".join(f"{s}: {len(st)} seq" for s, st in STORES.items()) + "; 20 random sequences per split re-read from the archive and identical")

with CHECKS("4. Coordinate and confidence validity") as chk:
    fails = [m for s_, m in flags if s_ == "FAIL"]
    assert not fails, "diagnostic FAIL flags: " + " | ".join(fails)
    conf = DIAG_REPORT["confidence"]
    assert conf["min"] is not None and conf["min"] >= 0 and conf["max"] <= 1 + 1e-6, f"confidence out of [0,1]: {conf}"
    total_valid, total_lm, outside = 0, 0, 0
    for s, st in STORES.items():
        assert float(max(m["max_quant_error"] for m in st.metas)) < 2e-3, f"{s}: storage precision loss too large"
        for i in range(len(st)):
            _, _, coords, valid = st.get(i)
            assert np.isfinite(coords).all(), f"{s}[{i}]: non-finite stored coordinates"
            lm = coords.reshape(-1, 75, 3)
            assert np.all(lm[~valid] == 0.0), f"{s}[{i}]: invalid landmarks are not stored as the fill value"
            total_valid += int(valid.sum())
            total_lm += valid.size
            xy = lm[..., :2][valid]
            outside += int((np.abs(xy) > 1.0).sum())
    assert total_valid > 0
    chk.note(f"{total_valid / total_lm:.0%} of stored landmarks valid; {outside / max(2 * total_valid, 1):.2%} of valid x/y outside [-1, 1] (kept, not clamped); "
             f"warnings: {[m for s_, m in flags if s_ == 'WARN'] or 'none'}")

## 10. Device, data loaders, configuration and model

In [ ]:
# CELL 13 — Import the patched repository modules and select the device.
def reset_imports():
    """Drop cached copies of repository / project modules so re-runs see the current files."""
    stems = {p.stem for d in (CODE_DIR, REPO_DIR) for p in d.glob("*.py") if not p.stem.startswith("__")}
    for s in stems - {"isign_pose", "isign_report", "repo_patches"}:
        sys.modules.pop(s, None)
    for d in (CODE_DIR, REPO_DIR):
        if str(d) not in sys.path:
            sys.path.insert(0, str(d))
    importlib.invalidate_caches()


reset_imports()
import torch
import isign_data as idata
import isign_train as it
from batch import Batch
from constants import EOS_TOKEN, PAD_TOKEN, UNK_TOKEN
from loss import RegLoss, build_element_mask
from model import build_model
from vocabulary import Vocabulary

DEVICE, DEVICE_INFO = it.select_device(require_cuda=False)
IS_CUDA = DEVICE.type == "cuda"
it.set_seed(SEED)
print(json.dumps(DEVICE_INFO, indent=1))
if not IS_CUDA:
    print("\nWARNING: no GPU in this runtime. Preprocessing and smoke tests run on the CPU, but the CUDA checks will be "
          "UNVERIFIED and the decision cannot be READY. Switch to a T4 runtime for the final run.")

In [ ]:
# CELL 14 — Datasets, vocabulary, configuration, loaders.
import yaml

train_set = idata.ISignShardDataset(PROC_ROOT / "train", NORM, max_frames=QUALITY["max_frames"])
val_set = idata.ISignShardDataset(PROC_ROOT / "val", NORM, max_frames=QUALITY["max_frames"])
test_set = idata.ISignShardDataset(PROC_ROOT / "test", NORM, max_frames=QUALITY["max_frames"])
pilot_ds = idata.ISignShardDataset(PILOT_DIR, NORM)
pilot_set = torch.utils.data.Subset(pilot_ds, list(range(min(PILOT_N, len(pilot_ds)))))

src_vocab = idata.build_vocab(train_set.token_lists())          # training split only
pad_id = src_vocab.stoi[PAD_TOKEN]
src_vocab.to_file(str(WORK_DIR / "src_vocab.txt"))

cfg = yaml.safe_load((REPO_DIR / "Configs" / "Base.yaml").read_text())
cfg["data"].update({"src": "text", "train": str(PROC_ROOT / "train"), "dev": str(PROC_ROOT / "val"), "test": str(PROC_ROOT / "test")})
cfg["data"].pop("src_vocab", None)
cfg["model"]["trg_size"] = ip.N_COORDS                           # 225 coordinates (+1 counter added by build_model)
cfg["training"].update({
    "use_cuda": bool(IS_CUDA), "batch_size": BATCH_SIZE, "epochs": MAX_EPOCHS, "learning_rate": LEARNING_RATE,
    "early_stop_patience": EARLY_STOP_PATIENCE, "val_every_epochs": 1, "continue": False, "overwrite": True,
    "model_dir": str(DRIVE_OUT / "runs" / "main"), "random_seed": SEED,
})
cfg["normalization"] = NORM                                      # saved next to the model configuration
cfg_path = WORK_DIR / "config.yaml"
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
(DRIVE_OUT / "config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
cfg = yaml.safe_load(cfg_path.read_text())

train_loader = idata.make_loader(train_set, src_vocab, BATCH_SIZE, shuffle=True, seed=SEED, num_workers=NUM_WORKERS)
val_loader = idata.make_loader(val_set, src_vocab, BATCH_SIZE, num_workers=NUM_WORKERS)
test_loader = idata.make_loader(test_set, src_vocab, BATCH_SIZE, num_workers=NUM_WORKERS)    # not used before model selection

print(f"train/val/test/pilot sequences: {len(train_set)}/{len(val_set)}/{len(test_set)}/{len(pilot_set)}")
print(f"source vocabulary (incl. 4 specials): {len(src_vocab)} | target width {cfg['model']['trg_size'] + 1} | batch size {BATCH_SIZE}")
print("config + normalisation saved to", cfg_path, "and", DRIVE_OUT / "config.yaml")

In [ ]:
# CELL 15 — Build the model for 225-coordinate iSign targets and put it on the device.
it.set_seed(SEED)
model = build_model(cfg, src_vocab=src_vocab, trg_vocab=Vocabulary(tokens=[]))
it.prepare_model(model, DEVICE)
criterion = RegLoss(cfg)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model on {it.parameter_devices(model)} | {n_params / 1e6:.1f} M parameters | loss: {cfg['training']['loss']}")
assert model.in_trg_size == ip.FRAME_WIDTH and model.out_trg_size == ip.FRAME_WIDTH
assert model.decoder.output_layer.out_features == ip.FRAME_WIDTH and model.trg_embed.in_features == ip.FRAME_WIDTH
assert model.src_embed.lut.num_embeddings == len(src_vocab)
print("No pretrained weights are loaded: the repository ships none, so there is nothing that could mismatch the iSign sizes.")

## 11. Checks (data representation, masks, model)

In [ ]:
# CELL 16 — Frame width / counter, decoder shift alignment, frame-level padding masks.
by_len = sorted(range(len(train_set)), key=lambda i: int(train_set.store.lengths[i]))
examples = [train_set[by_len[0]], train_set[by_len[len(by_len) // 2]], train_set[by_len[-1]]]      # shortest, median, longest -> real padding
raw = idata.collate_isign(examples, src_vocab)
batch = it.make_batch(raw, pad_id, model, DEVICE)

with CHECKS("5. Frame width and frame-counter consistency") as chk:
    for k in range(min(50, len(train_set))):
        item = train_set[k]
        frames, valid = item["trg"].numpy(), item["landmark_valid"].numpy()
        T = frames.shape[0]
        assert frames.shape == (T, 226) and valid.shape == (T, 75), f"seq {k}: bad shapes {frames.shape}/{valid.shape}"
        assert np.array_equal(frames[:, -1], (np.arange(T, dtype=np.float32) / np.float32(T))), f"seq {k}: counter is not frame_index / n_frames"
        assert frames[0, -1] == 0.0 and frames[-1, -1] < 1.0 and np.all(np.diff(frames[:, -1]) > 0)
    chk.note("226 = 225 coordinates + counter (frame_index / n_frames); checked on up to 50 sequences")

with CHECKS("5b. Decoder shift and target alignment") as chk:
    for b, ex in enumerate(examples):
        T = ex["trg"].shape[0]
        assert torch.equal(batch.trg_input[b, :T - 1].cpu(), ex["trg"][:-1]), f"seq {b}: decoder input is not frames[:-1]"
        assert torch.equal(batch.trg[b, :T - 1].cpu(), ex["trg"][1:]), f"seq {b}: target is not frames[1:]"
        assert torch.allclose(batch.trg[b, :T - 1, -1].cpu(), torch.arange(1, T, dtype=torch.float32) / T), "target counters misaligned"
        assert not batch.trg[b, T - 1:].any(), f"seq {b}: padded target frames are not zero"
    chk.note("input at step t is frame t; target at step t is frame t+1 (counter included); padding is zero")

with CHECKS("7. Frame-level padding masking") as chk:
    B, L = batch.trg.shape[0], batch.trg.shape[1]
    lengths = [ex["trg"].shape[0] for ex in examples]
    assert max(lengths) > min(lengths), "need sequences of different length to test padding"
    assert tuple(batch.trg_mask.shape) == (B, 1, L, L) and batch.trg_mask.dtype == torch.bool
    assert batch.frame_mask.sum(1).cpu().tolist() == [n - 1 for n in lengths], "frame mask != frame_length - 1"
    for b, n in enumerate(lengths):
        assert batch.trg_mask[b, 0, :, : n - 1].all() and not batch.trg_mask[b, 0, :, n - 1:].any(), f"seq {b}: decoder padding mask wrong"
    elem = build_element_mask(batch.trg, batch.frame_mask, batch.coord_mask)
    for b, n in enumerate(lengths):
        assert not elem[b, n - 1:].any(), f"seq {b}: padded frames can reach the loss"
    # padded frames must carry no gradient, whatever the predictions there are
    preds = torch.randn_like(batch.trg, requires_grad=True)
    loss_value = criterion(preds, batch.trg, frame_mask=batch.frame_mask, coord_mask=batch.coord_mask)
    loss_value.backward()
    for b, n in enumerate(lengths):
        assert torch.all(preds.grad[b, n - 1:] == 0), f"seq {b}: gradient on padded frames"
    chk.note(f"lengths {lengths}; decoder mask [B,1,T,T]; padded frames excluded from loss and gradient")
print("Batch tensors by device:", it.assert_batch_device(batch, DEVICE))

In [ ]:
# CELL 17 — Landmark-level masking: unit tests of the loss plus alignment on a real batch.
def run_loss_unit_tests(loss_name):
    local_cfg = copy.deepcopy(cfg)
    local_cfg["training"]["loss"] = loss_name
    crit = RegLoss(local_cfg)
    dev = DEVICE
    err = (lambda d: d * d) if loss_name.lower() == "mse" else (lambda d: abs(d))

    # a) valid zero-valued target coordinates DO contribute
    p = torch.tensor([[[1.0, 2.0]]], device=dev, requires_grad=True)
    t = torch.tensor([[[0.0, 2.0]]], device=dev)
    ones = torch.ones(1, 1, 2, dtype=torch.bool, device=dev)
    loss = crit(p, t, coord_mask=ones)
    loss.backward()
    assert abs(loss.item() - (err(1.0) + err(0.0)) / 2) < 1e-6, f"{loss_name}: valid zero target not counted"
    assert p.grad[0, 0, 0].item() != 0.0, f"{loss_name}: valid zero target produced no gradient"

    # b) invalid coordinates do NOT contribute, whatever the prediction there
    for bad_value in (1.0, 1000.0):
        p = torch.tensor([[[bad_value, 2.0]]], device=dev, requires_grad=True)
        mask = torch.tensor([[[False, True]]], device=dev)
        loss = crit(p, t, coord_mask=mask)
        loss.backward()
        assert abs(loss.item()) < 1e-6, f"{loss_name}: invalid coordinate leaked into the loss"
        assert p.grad[0, 0, 0].item() == 0.0, f"{loss_name}: gradient at an invalid coordinate"
        assert torch.isfinite(p.grad).all()

    # c) padded frame excluded; d) frame and coordinate masks combine (AND)
    p = torch.tensor([[[1.0, 1.0], [5.0, 5.0], [7.0, 7.0]]], device=dev, requires_grad=True)
    t3 = torch.zeros(1, 3, 2, device=dev)
    frame = torch.tensor([[True, True, False]], device=dev)
    coord = torch.tensor([[[True, True], [False, True], [True, True]]], device=dev)
    loss = crit(p, t3, frame_mask=frame, coord_mask=coord)
    loss.backward()
    expected = (err(1.0) + err(1.0) + err(5.0)) / 3        # frame0: 2 coords, frame1: only coord 1, frame2: padded
    assert abs(loss.item() - expected) < 1e-5, f"{loss_name}: combined mask value {loss.item()} != {expected}"
    assert torch.all(p.grad[0, 2] == 0) and p.grad[0, 1, 0].item() == 0.0 and p.grad[0, 1, 1].item() != 0.0
    assert torch.isfinite(p.grad).all() and torch.isfinite(loss)

    # e) a mask one column narrower than the targets leaves the counter column valid (and padded frames still excluded)
    p = torch.ones(1, 2, 3, device=dev, requires_grad=True)
    t0 = torch.zeros(1, 2, 3, device=dev)
    cm = torch.tensor([[[False, False], [False, False]]], device=dev)
    loss = crit(p, t0, frame_mask=torch.tensor([[True, False]], device=dev), coord_mask=cm)
    loss.backward()
    assert abs(loss.item() - err(1.0)) < 1e-6 and p.grad[0, 0, 2].item() != 0.0 and torch.all(p.grad[0, 1] == 0)

    # f) nothing valid at all: exactly zero, finite, no gradient, no division by zero
    p = torch.randn(2, 3, 4, device=dev, requires_grad=True)
    loss = crit(p, torch.randn(2, 3, 4, device=dev), frame_mask=torch.zeros(2, 3, dtype=torch.bool, device=dev))
    loss.backward()
    assert loss.item() == 0.0 and torch.all(p.grad == 0) and torch.isfinite(loss)
    all_bad = torch.zeros(2, 3, 4, dtype=torch.bool, device=dev)
    assert int(build_element_mask(torch.zeros(2, 3, 4, device=dev), coord_mask=all_bad).sum()) == 0

    # g) wrong shapes are rejected loudly
    for kwargs in ({"frame_mask": torch.ones(2, 4, dtype=torch.bool, device=dev)},
                   {"coord_mask": torch.ones(2, 3, 9, dtype=torch.bool, device=dev)}):
        try:
            crit(torch.zeros(2, 3, 4, device=dev), torch.zeros(2, 3, 4, device=dev), **kwargs)
            raise AssertionError("a wrongly shaped mask should have been rejected")
        except ValueError:
            pass


with CHECKS("6. Landmark-level masking") as chk:
    for _name in ("MSE", "L1"):
        run_loss_unit_tests(_name)
    # the same flags on a real batch: coordinate triplets line up with landmark flags, shifted by one frame
    lv = raw.landmark_valid
    assert tuple(batch.coord_mask.shape) == (lv.shape[0], lv.shape[1] - 1, 225)
    assert int(batch.coord_mask.sum()) == 3 * int(lv[:, 1:].sum()), "coordinate mask count != 3 x landmark flags"
    checked_invalid = checked_valid = 0
    for b, ex in enumerate(examples):
        T = ex["trg"].shape[0]
        for t_idx in range(0, T - 1, max(1, (T - 1) // 6)):
            for j in range(75):
                flag = bool(lv[b, t_idx + 1, j])
                triplet = batch.coord_mask[b, t_idx, 3 * j: 3 * j + 3].cpu().tolist()
                assert triplet == [flag] * 3, f"seq {b} step {t_idx} landmark {j}: {triplet} vs flag {flag}"
                checked_valid += flag
                checked_invalid += not flag
    assert checked_invalid > 0 and checked_valid > 0, "this batch has no invalid (or no valid) landmarks to check"
    # invalid landmarks are stored as 0.0; valid zero-valued targets (if any) keep their flag
    zero_but_valid = int(((batch.trg[..., :225] == 0) & batch.coord_mask).sum())
    chk.note(f"MSE and L1 unit tests passed; {checked_valid} valid / {checked_invalid} invalid landmark triplets verified on a real batch; "
             f"{zero_but_valid} valid zero-valued target coordinates in this batch")

In [ ]:
# CELL 18 — Forward pass, causality, finite loss and gradients.
with CHECKS("9. Model forward pass") as chk:
    model.eval()
    with torch.no_grad():
        pred, _ = model(src=batch.src, trg_input=batch.trg_input, src_mask=batch.src_mask,
                        src_lengths=batch.src_lengths, trg_mask=batch.trg_mask)
        assert pred.shape == batch.trg.shape, f"prediction {tuple(pred.shape)} vs target {tuple(batch.trg.shape)}"
        assert torch.isfinite(pred).all()
        # no look-ahead: changing future decoder inputs must not change earlier predictions
        k = batch.trg_input.shape[1] // 2
        altered = batch.trg_input.clone()
        altered[:, k:, :] += 1.0
        pred2, _ = model(src=batch.src, trg_input=altered, src_mask=batch.src_mask,
                         src_lengths=batch.src_lengths, trg_mask=batch.trg_mask)
        assert torch.allclose(pred[:, :k], pred2[:, :k], atol=1e-5), "earlier predictions changed: the decoder sees the future"
        assert not torch.allclose(pred[:, k:], pred2[:, k:], atol=1e-5), "later predictions ignore their inputs"
    # padding invariance: a sequence alone and padded inside a batch give the same predictions
    worst = 0.0
    for b, ex in enumerate(examples):
        raw_one = idata.collate_isign([ex], src_vocab)
        one = it.make_batch(raw_one, pad_id, model, DEVICE)
        with torch.no_grad():
            alone, _ = model(src=one.src, trg_input=one.trg_input, src_mask=one.src_mask, src_lengths=one.src_lengths, trg_mask=one.trg_mask)
        n = ex["trg"].shape[0] - 1
        worst = max(worst, (pred[b, :n] - alone[0, :n]).abs().max().item())
    assert worst < 1e-3, f"padded batch differs from the single-sequence run by {worst:.2e}"
    chk.note(f"output {tuple(pred.shape)}; causal; padding invariance max diff {worst:.1e}")

with CHECKS("10. Finite loss and gradients") as chk:
    # (a) the model's own loss uses both masks. Compared in eval mode: in training mode the repository's feed-forward
    #     layers keep a fixed dropout of 0.1 (the config's dropout 0.0 does not reach them), so two passes would differ.
    model.eval()
    with torch.no_grad():
        eval_loss, _ = model.get_loss_for_batch(batch, criterion)
        pred_eval = model(src=batch.src, trg_input=batch.trg_input, src_mask=batch.src_mask,
                          src_lengths=batch.src_lengths, trg_mask=batch.trg_mask)[0]
        manual = criterion(pred_eval, batch.trg, frame_mask=batch.frame_mask, coord_mask=batch.coord_mask)
        unmasked = criterion(pred_eval, batch.trg)
    assert abs(eval_loss.item() - manual.item()) < 1e-5 * max(1.0, abs(manual.item())), "get_loss_for_batch is not using both masks"
    assert abs(eval_loss.item() - unmasked.item()) > 1e-6, "masked and legacy losses are identical: the masks have no effect"
    # (b) training-mode forward / backward
    model.train()
    model.zero_grad(set_to_none=True)
    loss_value, _ = model.get_loss_for_batch(batch, criterion)
    assert torch.isfinite(loss_value), "non-finite loss"
    loss_value.backward()
    missing = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
    assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None), "non-finite gradient"
    norm = torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters() if p.grad is not None)).item()
    assert norm > 0, "all gradients are zero"
    model.zero_grad(set_to_none=True)
    model.eval()
    chk.note(f"loss {loss_value.item():.4f}, gradient norm {norm:.3f}" + (f"; parameters without gradient: {missing}" if missing else ""))

In [ ]:
# CELL 19 — Device placement and one full optimiser step (CUDA acceptance test).
with CHECKS("11. CUDA execution and optimizer update") as chk:
    print("device:", DEVICE, "| parameters on:", it.parameter_devices(model))
    assert it.parameter_devices(model) == [str(DEVICE)], f"model parameters on {it.parameter_devices(model)}, expected {DEVICE}"
    step_batch = it.make_batch(raw, pad_id, model, DEVICE)
    placed = it.assert_batch_device(step_batch, DEVICE)
    print("batch tensors:", placed)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    before = [p.detach().clone() for p in model.parameters()]
    result = it.train_step(model, step_batch, criterion, optimizer, cfg["training"].get("clip_grad_norm"))
    if IS_CUDA:
        torch.cuda.synchronize()
    assert result["skipped"] is None, f"step skipped: {result['skipped']}"
    assert np.isfinite(result["loss"]) and np.isfinite(result["grad_norm"])
    changed = sum(not torch.equal(b, p.detach()) for b, p in zip(before, model.parameters()))
    assert changed > 0.9 * len(before), f"only {changed}/{len(before)} parameter tensors changed"
    assert it.parameter_devices(model) == [str(DEVICE)], "a parameter left the device"
    detail = f"loss {result['loss']:.4f}, grad norm {result['grad_norm']:.3f}, {changed}/{len(before)} parameter tensors updated"
    if IS_CUDA:
        detail += f", peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.2f} GB at batch size {raw.trg.shape[0]}"
        chk.note(detail)
    else:
        chk.unverified("the step ran, but on the CPU: no CUDA device in this runtime. " + detail)
    del optimizer, before
    gc.collect()

## 12. Pilot regression (isolated 5-sequence set)

In [ ]:
# CELL 20 — The original pilot still passes through the same patched code (separate vocabulary and model).
with CHECKS("P. Pilot regression (isolated 5-sequence set)") as chk:
    pilot_vocab = idata.build_vocab(pilot_ds.token_lists())
    pilot_loader = idata.make_loader(pilot_set, pilot_vocab, PILOT_BATCH_SIZE)
    it.set_seed(SEED)
    pilot_model = it.prepare_model(build_model(cfg, src_vocab=pilot_vocab, trg_vocab=Vocabulary(tokens=[])), DEVICE)
    pilot_opt = torch.optim.Adam(pilot_model.parameters(), lr=LEARNING_RATE)
    stats = it.run_train_epoch(pilot_model, pilot_loader, criterion, pilot_opt, pilot_vocab.stoi[PAD_TOKEN], DEVICE, cfg["training"].get("clip_grad_norm"))
    assert stats["steps"] >= 1 and np.isfinite(stats["loss"]), stats
    chk.note(f"{len(pilot_set)} pilot sequences, one epoch: loss {stats['loss']:.4f}")
    del pilot_model, pilot_opt
    gc.collect()

## 13. Small-subset overfit experiment
Trains a **fresh** model on a small fixed subset of the **training** split for a few hundred steps. Validation and test data are not used. The outcome is reported as measured; there is no guaranteed threshold, and overfitting a tiny subset says nothing about generalisation or about meaningful signing.

In [ ]:
# CELL 21 — Overfit experiment.
OVERFIT = {}
with CHECKS("13. Small-subset overfit experiment") as chk:
    ready = CHECKS.all_passed(["5. Frame width and frame-counter consistency", "5b. Decoder shift and target alignment", "6. Landmark-level masking",
                               "7. Frame-level padding masking", "9. Model forward pass", "10. Finite loss and gradients"])
    assert ready, "earlier checks failed: diagnose those before the overfit experiment"
    lengths_all, vfrac_all = train_set.store.lengths, train_set.store.valid_fraction
    eligible = [i for i in range(len(train_set)) if lengths_all[i] <= OVERFIT_MAX_FRAMES]
    eligible.sort(key=lambda i: (-vfrac_all[i], i))
    subset_idx = sorted(eligible[:OVERFIT_N])
    assert len(subset_idx) >= 8, f"only {len(subset_idx)} eligible sequences (<= {OVERFIT_MAX_FRAMES} frames); convert more shards or raise OVERFIT_MAX_FRAMES"
    subset = torch.utils.data.Subset(train_set, subset_idx)
    sub_loader = idata.make_loader(subset, src_vocab, OVERFIT_BATCH, shuffle=True, seed=SEED)
    sub_eval = idata.make_loader(subset, src_vocab, OVERFIT_BATCH)

    it.set_seed(SEED)
    of_model = it.prepare_model(build_model(cfg, src_vocab=src_vocab, trg_vocab=Vocabulary(tokens=[])), DEVICE)
    before = it.evaluate(of_model, sub_eval, criterion, pad_id, DEVICE)
    print(f"subset: {len(subset_idx)} sequences, mean valid fraction {vfrac_all[subset_idx].mean():.2f}, "
          f"frames {int(lengths_all[subset_idx].min())}-{int(lengths_all[subset_idx].max())}; loss before training {before['loss']:.4f}")
    history = it.overfit_steps(of_model, sub_loader, criterion, cfg, pad_id, DEVICE, OVERFIT_STEPS, log_every=25)
    after = it.evaluate(of_model, sub_eval, criterion, pad_id, DEVICE)

    done = [h for h in history if not h["skipped"]]
    skipped = len(history) - len(done)
    losses = np.array([h["loss"] for h in done])
    grads = np.array([h["grad_norm"] for h in done])
    k = max(5, len(losses) // 10)
    first, last = float(losses[:k].mean()), float(losses[-k:].mean())
    OVERFIT.update({"subset_size": len(subset_idx), "steps": len(history), "skipped_steps": skipped, "loss_first_window": first,
                    "loss_last_window": last, "eval_loss_before": before["loss"], "eval_loss_after": after["loss"],
                    "max_grad_norm": float(grads.max()), "window": k})
    print(json.dumps(OVERFIT, indent=1))
    assert skipped == 0, f"{skipped} steps were skipped (non-finite loss or gradient)"
    assert np.isfinite(losses).all() and np.isfinite(grads).all()
    assert grads.max() < 1e4, f"exploding gradients (max norm {grads.max():.1f})"
    assert last < first, f"loss did not decrease (first {first:.4f} -> last {last:.4f}): diagnose before any full training"
    chk.note(f"train loss {first:.4f} -> {last:.4f} (mean of first/last {k} steps); eval-mode loss on the subset {before['loss']:.4f} -> {after['loss']:.4f}; "
             f"max grad norm {grads.max():.2f}")
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6, 3.2))
        plt.plot(losses)
        plt.xlabel("optimizer step")
        plt.ylabel("masked loss")
        plt.title("Overfit experiment (training subset only)")
        plt.tight_layout()
        plt.savefig(DRIVE_OUT / "diagnostics" / "overfit_loss.png", dpi=110)
        plt.show()
    except Exception as exc:
        print("plot skipped:", exc)

In [ ]:
# CELL 22 — Checkpoint save / reload equivalence, and resume of the training loop.
with CHECKS("12. Checkpoint save/reload") as chk:
    source_model = of_model if "of_model" in globals() else model
    ckpt_dir = WORK_DIR / "ckpt_test"
    ckpt_dir.mkdir(exist_ok=True)
    ckpt_path = ckpt_dir / "roundtrip.ckpt"
    it.save_checkpoint(ckpt_path, source_model, state={"epoch": 0}, normalization=NORM, cfg=cfg)
    reloaded = it.prepare_model(build_model(cfg, src_vocab=src_vocab, trg_vocab=Vocabulary(tokens=[])), DEVICE)
    payload = it.load_checkpoint(ckpt_path, DEVICE)
    reloaded.load_state_dict(payload["model_state"])
    assert payload["normalization"] == NORM, "normalisation stored in the checkpoint differs"
    assert payload["config"]["model"]["trg_size"] == 225
    source_model.eval()
    reloaded.eval()
    probe = it.make_batch(raw, pad_id, source_model, DEVICE)
    with torch.no_grad():
        a, _ = source_model(src=probe.src, trg_input=probe.trg_input, src_mask=probe.src_mask, src_lengths=probe.src_lengths, trg_mask=probe.trg_mask)
        b, _ = reloaded(src=probe.src, trg_input=probe.trg_input, src_mask=probe.src_mask, src_lengths=probe.src_lengths, trg_mask=probe.trg_mask)
    assert torch.allclose(a, b, atol=1e-6), f"reloaded model differs by {(a - b).abs().max().item():.2e}"
    chk.note(f"max prediction difference {(a - b).abs().max().item():.1e} ({'trained' if 'of_model' in globals() else 'untrained'} weights)")

with CHECKS("12b. Training-loop resume and test-set guard") as chk:
    assert "subset" in globals() and CHECKS.status("13. Small-subset overfit experiment") == rep.PASS, "needs the overfit subset"
    mcfg = copy.deepcopy(cfg)
    mcfg["training"].update({"early_stop_patience": 50, "patience": 50, "learning_rate_min": 0.0})
    mdir = WORK_DIR / "machinery_test"
    if mdir.exists():
        shutil.rmtree(mdir)
    # machinery only: the training subset doubles as 'validation' here, so these numbers mean nothing
    it.set_seed(SEED)
    m1 = it.prepare_model(build_model(cfg, src_vocab=src_vocab, trg_vocab=Vocabulary(tokens=[])), DEVICE)
    s1 = it.train(m1, sub_loader, sub_eval, criterion, mcfg, mdir, DEVICE, pad_id, NORM, max_epochs=2, resume=True, log=lambda m: None)
    assert s1["epoch"] == 2 and (mdir / "last.ckpt").exists() and (mdir / "best.ckpt").exists() and (mdir / "selection_done.json").exists()
    # simulate a Colab interruption: new process-like state, resume to 4 epochs
    it.set_seed(SEED + 1)
    m2 = it.prepare_model(build_model(cfg, src_vocab=src_vocab, trg_vocab=Vocabulary(tokens=[])), DEVICE)
    s2 = it.train(m2, sub_loader, sub_eval, criterion, mcfg, mdir, DEVICE, pad_id, NORM, max_epochs=4, resume=True, log=lambda m: None)
    assert s2["epoch"] == 4 and len(s2["history"]) == 4 and s2["history"][:2] == s1["history"][:2], "resume did not continue from the saved state"
    assert s2["best_epoch"] >= 1
    # the test set must be unreachable before model selection finished
    try:
        it.final_test_evaluation(m2, sub_eval, criterion, WORK_DIR / "never_trained", DEVICE, pad_id)
        raise AssertionError("test evaluation was allowed without finished model selection")
    except RuntimeError:
        pass
    chk.note("2 epochs, simulated interruption, resumed to 4 (history preserved); test evaluation refuses to run before selection is done")

with CHECKS("14. Free-running (greedy) evaluation") as chk:
    assert "of_model" in globals() and CHECKS.status("13. Small-subset overfit experiment") == rep.PASS, "needs the overfit model"
    teacher_forced = it.evaluate(of_model, sub_eval, criterion, pad_id, DEVICE, free_running=False, max_batches=1)
    free_running = it.evaluate(of_model, sub_eval, criterion, pad_id, DEVICE, free_running=True, max_batches=1)
    assert free_running["batches"] == 1 and np.isfinite(free_running["loss"]), free_running
    assert np.isfinite(teacher_forced["loss"])
    chk.note(f"one batch of the training subset: teacher-forced loss {teacher_forced['loss']:.4f}, free-running (ground-truth counters) "
             f"loss {free_running['loss']:.4f}; generated frames have the target shape")

## 14. Report and decision

In [ ]:
# CELL 23 — Pass/fail report, final representation, split statistics, decision.
REQUIRED = [
    "1. Dataset archive access (no combined archive)", "2. UID-to-pose matching", "3. Text/pose/file count alignment",
    "4. Coordinate and confidence validity", "5. Frame width and frame-counter consistency", "6. Landmark-level masking",
    "7. Frame-level padding masking", "8. Train/val/test separation by video ID", "9. Model forward pass",
    "10. Finite loss and gradients", "11. CUDA execution and optimizer update", "12. Checkpoint save/reload",
    "13. Small-subset overfit experiment", "5b. Decoder shift and target alignment", "12b. Training-loop resume and test-set guard",
    "14. Free-running (greedy) evaluation", "P. Pilot regression (isolated 5-sequence set)",
]
CHECKS.print_table(REQUIRED)
DECISION, BLOCKERS = rep.decide(CHECKS, REQUIRED)

split_stats = {}
for s in SPLITS:
    m, st = manifests[s], STORES[s]
    split_stats[s] = {"planned_sequences": len(m), "planned_videos": int(m["video_id"].nunique()),
                      "planned_share": round(len(m) / len(matched), 4),
                      "converted_sequences": len(st), "converted_videos": len({ip.video_id_of(u) for u in st.uids}),
                      "excluded_sequences": len(st.exclusions())}
representation = {
    "landmarks": "33 body + 21 left hand + 21 right hand (pose-format indices 0:33 and 501:543); layout verified from the file header: " + layout_note,
    "frame": "225 coordinates (landmark-major x,y,z) + 1 frame counter = 226",
    "frame_counter": "frame_index / n_frames (first frame 0.0, last (n-1)/n); regenerated at load time",
    "decoder_alignment": "input step t = frame t, target step t = frame t+1 (counter included); input and target are shifted by one frame",
    "invalid_landmarks": "stored as 0.0 and flagged in a separate [T,75] validity array; the loss masks them (and padded frames)",
    "validity_rule": f"valid = finite and not masked and confidence >= {CONF_THRESHOLD}" + (" and not an exact raw (0,0,0)" if ZERO_FILL_IS_MISSING else ""),
    "normalization": NORM, "normalization_hash": NORM_HASH,
    "no_imputation": "missing landmarks are not interpolated; decoder inputs contain 0.0 at those positions",
}
sections = {
    "Final coordinate representation": representation,
    "Split statistics (by video, seed %d)" % SEED: split_stats,
    "Conversion": [{k: v for k, v in r.items()} for r in conv_rows],
    "Diagnostics flags": [{"severity": s_, "message": m_} for s_, m_ in flags],
    "Overfit outcome": OVERFIT,
    "Runtime": DEVICE_INFO,
    "Not verified / limitations": (
        "Unconstrained inference (text only, no ground-truth counters) is not part of this notebook; greedy decoding uses the "
        "ground-truth frame counters as in the original repository. Resume granularity is one epoch. The text-format .skels of the "
        "pilot was replaced by binary shards. Test-set performance has not been measured."),
}
report_md = rep.render_markdown("iSign text-to-pose: pre-training readiness report", DECISION, BLOCKERS, CHECKS, sections)
(WORK_DIR / "pre_training_report.md").write_text(report_md, encoding="utf-8")
(DRIVE_OUT / "pre_training_report.md").write_text(report_md, encoding="utf-8")
(DRIVE_OUT / "pre_training_report.json").write_text(json.dumps({"decision": DECISION, "blockers": BLOCKERS, "checks": CHECKS.results,
                                                                "sections": sections}, indent=1, default=str), encoding="utf-8")
print("\n" + "=" * 70)
print("DECISION:", DECISION)
for b in BLOCKERS:
    print(f"  - {b['check']}: {b['status']}  {b['detail']}")
print("=" * 70)
print("report saved to", DRIVE_OUT / "pre_training_report.md")

## 15. Guarded actions — nothing below runs unless you enable it
Full training and the test-set evaluation are off by default. Review the report first; then set `RUN_FULL_TRAINING = True` (cell 1) and run the first cell. The test-set cell only works after training has finished (`selection_done.json` must exist) and only once.

In [ ]:
# CELL 24 — Full training (guarded).
if not RUN_FULL_TRAINING:
    print("RUN_FULL_TRAINING is False — not starting. Review the report above first.")
elif DECISION != "READY FOR CONTROLLED TRAINING":
    print("Decision is", DECISION, "— not starting. Resolve the blockers first.")
else:
    if REQUIRE_GPU_FOR_TRAINING and not IS_CUDA:
        raise RuntimeError("Training requires a GPU (Runtime > Change runtime type > T4 GPU).")
    it.set_seed(SEED)
    train_model = it.prepare_model(build_model(cfg, src_vocab=src_vocab, trg_vocab=Vocabulary(tokens=[])), DEVICE)
    final_state = it.train(train_model, train_loader, val_loader, criterion, cfg, DRIVE_OUT / "runs" / "main",
                           DEVICE, pad_id, NORM, max_epochs=MAX_EPOCHS, resume=True)
    print("stopped:", final_state["stop_reason"], "| best epoch", final_state["best_epoch"], "| best val", final_state["best_val"])

In [ ]:
# CELL 25 — Test-set evaluation (guarded; only after training finished; only once).
if not RUN_TEST_EVAL:
    print("RUN_TEST_EVAL is False — the test split has not been touched.")
else:
    results = it.final_test_evaluation(train_model if "train_model" in globals() else model, test_loader, criterion,
                                       DRIVE_OUT / "runs" / "main", DEVICE, pad_id, free_running=True)
    print(json.dumps(results, indent=1))